# Setting Directories

In [1]:
wd = "/home/manndo/master_dev"
autodock_poses_dir = wd + "/docking_ready_mgltools/docking"
diffdock_poses_dir = wd + "/diffdock_results"
equibind_poses_dir = wd + "/equibind_pocket_guided"

# Setting proteins_dir to the path of the Orai proteins directory for PoseBuster Mode "dock"
proteins_dir = wd + "/Orai"

In [2]:
"""
Overview: Pose Counts by Docking Method and Protein-Ligand Combination
This cell analyzes all docking results directories and creates a summary table
"""
import pandas as pd
from pathlib import Path
import re

def count_autodock_poses():
    """Count AutoDock Vina poses from PDBQT files"""
    autodock_data = []
    base_path = Path(autodock_poses_dir)
    
    for pdbqt_file in base_path.glob("*_vina_out.pdbqt"):
        stem = pdbqt_file.stem.replace("_vina_out", "")
        parts = stem.split("__")
        
        if len(parts) == 2:
            protein, ligand = parts
            
            # Count MODEL entries in PDBQT file
            with open(pdbqt_file, 'r') as f:
                content = f.read()
                pose_count = content.count('MODEL')
                if pose_count == 0:
                    pose_count = 1  # Single pose file without MODEL markers
            
            autodock_data.append({
                "Protein": protein,
                "Ligand": ligand,
                "AutoDock_Vina": pose_count
            })
    
    return pd.DataFrame(autodock_data)


def count_diffdock_poses():
    """Count DiffDock poses from SDF files in subdirectories"""
    diffdock_data = []
    base_path = Path(diffdock_poses_dir)
    
    for subdir in base_path.iterdir():
        if not subdir.is_dir():
            continue
        # Skip helper directories
        if "_ligand__" in subdir.name or subdir.name in ["prepared_proteins", "converted_pdbqt"]:
            continue
        
        parts = subdir.name.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            
            # Count all SDF files in subdirectory (including complex_* folders)
            pose_count = len(list(subdir.glob("**/*.sdf")))
            
            if pose_count > 0:
                diffdock_data.append({
                    "Protein": protein,
                    "Ligand": ligand,
                    "DiffDock": pose_count
                })
    
    return pd.DataFrame(diffdock_data)


def count_equibind_poses():
    """Count EquiBind poses from SDF files, supporting spatial_sites directory structure.
    
    Handles directories named like:
      {ligand}__{protein}_spatial_sites/site_XX/pose_XX.sdf
    as well as flat directories:
      {ligand}__{protein}/*.sdf
    """
    equibind_data = []
    base_path = Path(equibind_poses_dir)
    
    for subdir in base_path.iterdir():
        if not subdir.is_dir():
            continue
        # Skip pymol export/fixed variants for main count
        if "_pymol" in subdir.name:
            continue
        
        dir_name = subdir.name
        
        # Handle spatial_sites suffix: strip it to extract the protein name
        is_spatial = dir_name.endswith("_spatial_sites")
        if is_spatial:
            dir_name_clean = dir_name.replace("_spatial_sites", "")
        else:
            dir_name_clean = dir_name
        
        parts = dir_name_clean.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            
            # Count SDF files recursively (covers site_XX subdirs and flat layouts)
            pose_count = len(list(subdir.glob("**/*.sdf")))
            
            if pose_count > 0:
                equibind_data.append({
                    "Protein": protein,
                    "Ligand": ligand,
                    "EquiBind": pose_count
                })
    
    return pd.DataFrame(equibind_data)


# Collect pose counts from all methods
print("=" * 100)
print("DOCKING POSES OVERVIEW")
print("=" * 100)

print("\nCollecting pose counts from all docking methods...\n")

df_autodock = count_autodock_poses()
df_diffdock = count_diffdock_poses()
df_equibind = count_equibind_poses()

print(f"  AutoDock Vina: {len(df_autodock)} protein-ligand combinations")
print(f"  DiffDock:      {len(df_diffdock)} protein-ligand combinations")
print(f"  EquiBind:      {len(df_equibind)} protein-ligand combinations")

# Merge all three methods into one table
if not df_autodock.empty:
    df_combined = df_autodock
    if not df_diffdock.empty:
        df_combined = df_combined.merge(df_diffdock, on=["Protein", "Ligand"], how="outer")
    if not df_equibind.empty:
        df_combined = df_combined.merge(df_equibind, on=["Protein", "Ligand"], how="outer")
elif not df_diffdock.empty:
    df_combined = df_diffdock
    if not df_equibind.empty:
        df_combined = df_combined.merge(df_equibind, on=["Protein", "Ligand"], how="outer")
else:
    df_combined = df_equibind if not df_equibind.empty else pd.DataFrame()

if not df_combined.empty:
    # Fill NaN with 0 and convert to int
    df_combined = df_combined.fillna(0)
    for col in ["AutoDock_Vina", "DiffDock", "EquiBind"]:
        if col in df_combined.columns:
            df_combined[col] = df_combined[col].astype(int)
    
    # Add total column
    method_cols = [c for c in ["AutoDock_Vina", "DiffDock", "EquiBind"] if c in df_combined.columns]
    df_combined["Total"] = df_combined[method_cols].sum(axis=1)
    
    # Sort by protein, then ligand
    df_combined = df_combined.sort_values(["Protein", "Ligand"])
    
    # Display full table
    print("\n" + "=" * 100)
    print("POSE COUNTS BY PROTEIN-LIGAND COMBINATION")
    print("=" * 100)
    print(df_combined.to_string(index=False))
    
    # Summary statistics
    print("\n" + "=" * 100)
    print("SUMMARY STATISTICS")
    print("=" * 100)
    print(f"\nTotal unique protein-ligand combinations: {len(df_combined)}")
    print(f"\nTotal poses by method:")
    for col in method_cols:
        print(f"  {col}: {df_combined[col].sum():,} poses")
    print(f"  Combined Total: {df_combined['Total'].sum():,} poses")
    
    print(f"\nUnique proteins: {df_combined['Protein'].nunique()}")
    print(f"Unique ligands: {df_combined['Ligand'].nunique()}")
    
    # Breakdown by ligand
    print("\n" + "-" * 100)
    print("POSES BY LIGAND")
    print("-" * 100)
    ligand_summary = df_combined.groupby("Ligand")[method_cols + ["Total"]].sum()
    print(ligand_summary.to_string())
    
    # Breakdown by protein
    print("\n" + "-" * 100)
    print("POSES BY PROTEIN")
    print("-" * 100)
    protein_summary = df_combined.groupby("Protein")[method_cols + ["Total"]].sum()
    print(protein_summary.to_string())
    
    # Save to CSV
    output_file = Path( wd + "/posebusters_results") / "pose_counts_overview.csv"
    output_file.parent.mkdir(exist_ok=True)
    df_combined.to_csv(output_file, index=False)
    print(f"\n" + "=" * 100)
    print(f"Table saved to: {output_file}")
    print("=" * 100)
    
else:
    print("\nNo docking results found!")

DOCKING POSES OVERVIEW


  AutoDock Vina: 32 protein-ligand combinations
  DiffDock:      0 protein-ligand combinations
  EquiBind:      30 protein-ligand combinations

POSE COUNTS BY PROTEIN-LIGAND COMBINATION
                     Protein               Ligand  AutoDock_Vina  EquiBind  Total
        Orai1WT-MDSnap-Fr300         2abp-nh2-OPT             30       183    213
        Orai1WT-MDSnap-Fr300        2abp-nh3p-OPT              0       141    141
        Orai1WT-MDSnap-Fr300 Synta-66-OPT-Singlet             30       141    171
        Orai1WT-MDSnap-Fr300  gsk7975a-deprot-OPT             30       141    171
        Orai1WT-MDSnap-Fr300    gsk7975a-prot-OPT             30       141    171
Orai1WT-MDSnap-Fr300_cleaned         2abp-nh2-OPT             30       462    492
Orai1WT-MDSnap-Fr300_cleaned        2abp-nh3p-OPT              0       241    241
Orai1WT-MDSnap-Fr300_cleaned Synta-66-OPT-Singlet             30       241    271
Orai1WT-MDSnap-Fr300_cleaned  gsk7975a-deprot-OPT  

In [3]:
"""
Filter Poses: Keep Only Protein-Ligand Combinations Docked in ALL THREE Methods
=================================================================================
This cell filters the results to include only protein-ligand combinations that have
poses from AutoDock_Vina, DiffDock, AND EquiBind.
"""

# ============================================================================
# BUILD filtered_poses_df FROM df_combined (previous cell)
# ============================================================================
# Each row = one (docking_tool, protein, ligand, file_path, pose_count) entry

rows = []

# AutoDock Vina
base_path = Path(autodock_poses_dir)
for pdbqt_file in base_path.glob("*_vina_out.pdbqt"):
    stem = pdbqt_file.stem.replace("_vina_out", "")
    parts = stem.split("__")
    if len(parts) == 2:
        protein, ligand = parts
        with open(pdbqt_file, 'r') as f:
            content = f.read()
            pose_count = content.count('MODEL')
            if pose_count == 0:
                pose_count = 1
        rows.append({
            "docking_tool": "AutoDock_Vina",
            "protein": protein,
            "ligand": ligand,
            "file_path": str(pdbqt_file),
            "pose_count": pose_count
        })

# DiffDock
base_path = Path(diffdock_poses_dir)
for subdir in base_path.iterdir():
    if not subdir.is_dir():
        continue
    if "_ligand__" in subdir.name or subdir.name in ["prepared_proteins", "converted_pdbqt"]:
        continue
    parts = subdir.name.split("__")
    if len(parts) == 2:
        ligand, protein = parts
        pose_count = len(list(subdir.glob("**/*.sdf")))
        if pose_count > 0:
            rows.append({
                "docking_tool": "DiffDock",
                "protein": protein,
                "ligand": ligand,
                "file_path": str(subdir),
                "pose_count": pose_count
            })

# EquiBind
base_path = Path(equibind_poses_dir)
for subdir in base_path.iterdir():
    if not subdir.is_dir() or "_pymol" in subdir.name:
        continue
    
    dir_name = subdir.name
    # Handle spatial_sites suffix: strip it to extract the protein name
    is_spatial = dir_name.endswith("_spatial_sites")
    if is_spatial:
        dir_name_clean = dir_name.replace("_spatial_sites", "")
    else:
        dir_name_clean = dir_name
    
    parts = dir_name_clean.split("__")
    if len(parts) == 2:
        ligand, protein = parts
        # Count SDF files recursively (covers site_XX subdirs and flat layouts)
        pose_count = len(list(subdir.glob("**/*.sdf")))
        if pose_count > 0:
            rows.append({
                "docking_tool": "EquiBind",
                "protein": protein,
                "ligand": ligand,
                "file_path": str(subdir),
                "pose_count": pose_count
            })

filtered_poses_df = pd.DataFrame(rows)
print(f"Total entries: {len(filtered_poses_df)}")
print(f"Methods found: {filtered_poses_df['docking_tool'].unique().tolist()}")

# Find protein-ligand combinations present in all three methods
if not filtered_poses_df.empty:
    # Get unique combinations per method
    combos_by_method = {}
    for method in filtered_poses_df["docking_tool"].unique():
        method_df = filtered_poses_df[filtered_poses_df["docking_tool"] == method]
        combos = set(zip(method_df["protein"], method_df["ligand"]))
        combos_by_method[method] = combos
    
    print("=" * 100)
    print("FILTERING: Keep Only Protein-Ligand Combinations in ALL THREE Methods")
    print("=" * 100)
    
    print(f"\nProtein-ligand combinations per method:")
    for method, combos in combos_by_method.items():
        print(f"  {method}: {len(combos)} combinations")
    
    # Find intersection: combinations present in all three methods
    if len(combos_by_method) == 3:
        all_methods = list(combos_by_method.keys())
        common_combos = combos_by_method[all_methods[0]].intersection(
            combos_by_method[all_methods[1]], 
            combos_by_method[all_methods[2]]
        )
    elif len(combos_by_method) == 2:
        all_methods = list(combos_by_method.keys())
        common_combos = combos_by_method[all_methods[0]].intersection(
            combos_by_method[all_methods[1]]
        )
    else:
        common_combos = set()
    
    print(f"\nProtein-ligand combinations in ALL methods: {len(common_combos)}")
    
    if common_combos:
        print("\nCommon combinations:")
        for protein, ligand in sorted(common_combos):
            print(f"  {protein} + {ligand}")
        
        # Filter poses to only these combinations
        filtered_poses_df_all_three = filtered_poses_df[
            filtered_poses_df.apply(
                lambda row: (row["protein"], row["ligand"]) in common_combos,
                axis=1
            )
        ].copy()
        
        print(f"\n{'-' * 100}")
        print("FILTERED SUBSET (All Three Methods Only)")
        print(f"{'-' * 100}")
        
        # Show breakdown
        for method in filtered_poses_df_all_three["docking_tool"].unique():
            method_df = filtered_poses_df_all_three[filtered_poses_df_all_three["docking_tool"] == method]
            print(f"\n{method}:")
            print(f"  Combinations: {len(method_df)}")
            print(f"  Total poses: {method_df['pose_count'].sum():,}")
        
        # Make available for downstream cells
        filtered_poses_df = filtered_poses_df_all_three
        
        print(f"\n{'=' * 100}")
        print("filtered_poses_df has been updated to contain only combinations with all three methods")
        print(f"{'=' * 100}")

        # =================================================================
        # DETAILED SUMMARY: Poses per combination per docking method
        # =================================================================
        print(f"\n{'=' * 100}")
        print("DETAILED SUMMARY: Poses Kept per Combination per Docking Method")
        print(f"{'=' * 100}")

        # Build a pivot-style summary
        methods_in_df = sorted(filtered_poses_df["docking_tool"].unique())
        header = f"{'Protein':<35} {'Ligand':<30}" + "".join(f" {m:>15}" for m in methods_in_df) + f" {'Total':>10}"
        print(f"\n{header}")
        print("-" * len(header))

        grand_totals = {m: 0 for m in methods_in_df}
        grand_total_all = 0

        for protein, ligand in sorted(common_combos):
            row_str = f"{protein:<35} {ligand:<30}"
            row_total = 0
            for method in methods_in_df:
                count = filtered_poses_df[
                    (filtered_poses_df["protein"] == protein) &
                    (filtered_poses_df["ligand"] == ligand) &
                    (filtered_poses_df["docking_tool"] == method)
                ]["pose_count"].sum()
                row_str += f" {count:>15,}"
                row_total += count
                grand_totals[method] += count
            row_str += f" {row_total:>10,}"
            grand_total_all += row_total
            print(row_str)

        # Print totals row
        print("-" * len(header))
        totals_str = f"{'TOTAL':<35} {'':<30}"
        for method in methods_in_df:
            totals_str += f" {grand_totals[method]:>15,}"
        totals_str += f" {grand_total_all:>10,}"
        print(totals_str)

        print(f"\nFinal filtered_poses_df shape: {filtered_poses_df.shape}")
        print(f"Unique combinations kept: {len(common_combos)}")
        print(f"Total poses across all methods: {grand_total_all:,}")
    else:
        print("\nWARNING: No protein-ligand combinations found in all three methods!")
        print("Cannot filter further. Keeping current filtered_poses_df.")
else:
    print("filtered_poses_df is empty. Run the filtering cell first.")

Total entries: 62
Methods found: ['AutoDock_Vina', 'EquiBind']
FILTERING: Keep Only Protein-Ligand Combinations in ALL THREE Methods

Protein-ligand combinations per method:
  AutoDock_Vina: 32 combinations
  EquiBind: 30 combinations

Protein-ligand combinations in ALL methods: 24

Common combinations:
  Orai1WT-MDSnap-Fr300 + 2abp-nh2-OPT
  Orai1WT-MDSnap-Fr300 + Synta-66-OPT-Singlet
  Orai1WT-MDSnap-Fr300 + gsk7975a-deprot-OPT
  Orai1WT-MDSnap-Fr300 + gsk7975a-prot-OPT
  Orai1WT-MDSnap-Fr300_cleaned + 2abp-nh2-OPT
  Orai1WT-MDSnap-Fr300_cleaned + Synta-66-OPT-Singlet
  Orai1WT-MDSnap-Fr300_cleaned + gsk7975a-deprot-OPT
  Orai1WT-MDSnap-Fr300_cleaned + gsk7975a-prot-OPT
  Orai1WT-MDSnap-Fr400 + 2abp-nh2-OPT
  Orai1WT-MDSnap-Fr400 + Synta-66-OPT-Singlet
  Orai1WT-MDSnap-Fr400 + gsk7975a-deprot-OPT
  Orai1WT-MDSnap-Fr400 + gsk7975a-prot-OPT
  Orai1WT-MDSnap-Fr400_cleaned + 2abp-nh2-OPT
  Orai1WT-MDSnap-Fr400_cleaned + Synta-66-OPT-Singlet
  Orai1WT-MDSnap-Fr400_cleaned + gsk7975a-depro

In [4]:
"""
PoseBusters Analysis - Validate docked poses from the FILTERED SUBSET only
Checks: molecule validity, stereochemistry, bond geometry, ring flatness, etc.

NOTE: This cell uses the filtered_poses_df from the previous cell to only validate
poses that match the defined prefix/postfix filter criteria.
"""
import os
import subprocess
import re
import pandas as pd
from pathlib import Path
from posebusters import PoseBusters

# ============================================================================
# DISCOVER AVAILABLE PROTEIN FILES
# ============================================================================
proteins_base = Path(proteins_dir)
all_protein_pdbs = sorted(proteins_base.glob("**/*.pdb"))

print("=" * 80)
print("AVAILABLE PROTEIN PDB FILES")
print("=" * 80)
for pdb in all_protein_pdbs:
    print(f"  {pdb.relative_to(proteins_base)}")
print(f"\nTotal: {len(all_protein_pdbs)} PDB files found in {proteins_base}")


def find_protein_file(protein_name: str) -> str | None:
    """
    Find the PDB file for a given protein name by searching the proteins directory.
    
    Tries multiple matching strategies:
    1. Exact filename match (protein_name.pdb)
    2. Common suffixed patterns (_protein.pdb, _clean.pdb, _receptor.pdb)
    3. Recursive glob for any PDB containing the protein name
    4. Fuzzy matching: check if protein_name is a substring of any PDB filename
    
    Args:
        protein_name: The protein identifier (e.g., "Orai1WT-MDSnap-Fr400")
        
    Returns:
        Path to the protein PDB file as string, or None if not found
    """
    # Strategy 1: Exact match
    exact_candidates = [
        proteins_base / f"{protein_name}.pdb",
        proteins_base / f"{protein_name}_protein.pdb",
        proteins_base / f"{protein_name}_clean.pdb",
        proteins_base / f"{protein_name}_receptor.pdb",
    ]
    for candidate in exact_candidates:
        if candidate.exists():
            return str(candidate)
    
    # Strategy 2: Recursive exact match
    for candidate_name in [
        f"{protein_name}.pdb",
        f"{protein_name}_protein.pdb",
        f"{protein_name}_clean.pdb",
        f"{protein_name}_receptor.pdb",
    ]:
        matches = list(proteins_base.glob(f"**/{candidate_name}"))
        if matches:
            return str(matches[0])
    
    # Strategy 3: Recursive glob with protein name as prefix
    matches = list(proteins_base.glob(f"**/{protein_name}*.pdb"))
    if matches:
        # Prefer shortest filename (most likely the main structure)
        matches.sort(key=lambda p: len(p.name))
        return str(matches[0])
    
    # Strategy 4: Fuzzy - check if protein_name is contained in any PDB filename
    for pdb_file in all_protein_pdbs:
        if protein_name.lower() in pdb_file.stem.lower():
            return str(pdb_file)
    
    # Strategy 5: Try matching without common suffixes/prefixes
    # e.g., "Orai1WT-MDSnap-Fr400" might match "Orai1WT_MDSnap_Fr400.pdb"
    protein_name_normalized = protein_name.replace("-", "_").lower()
    for pdb_file in all_protein_pdbs:
        pdb_stem_normalized = pdb_file.stem.replace("-", "_").lower()
        if protein_name_normalized == pdb_stem_normalized:
            return str(pdb_file)
        if protein_name_normalized in pdb_stem_normalized:
            return str(pdb_file)
    
    return None


# Pre-build a protein file lookup cache for all unique protein names
# This avoids repeated filesystem searches
print("\n" + "-" * 80)
print("PROTEIN FILE RESOLUTION")
print("-" * 80)

_protein_file_cache = {}

if 'filtered_poses_df' in dir() and not filtered_poses_df.empty:
    unique_proteins = filtered_poses_df["protein"].unique()
    for pname in sorted(unique_proteins):
        pfile = find_protein_file(pname)
        _protein_file_cache[pname] = pfile
        status = f"✓ {Path(pfile).name}" if pfile else "✗ NOT FOUND"
        print(f"  {pname}: {status}")
    
    n_found = sum(1 for v in _protein_file_cache.values() if v is not None)
    n_missing = sum(1 for v in _protein_file_cache.values() if v is None)
    print(f"\n  Resolved: {n_found}/{len(unique_proteins)} proteins")
    if n_missing > 0:
        print(f"  WARNING: {n_missing} proteins have no matching PDB file!")
        print(f"  These poses will fall back to 'mol' mode (no intermolecular checks)")

# ============================================================================
# PDBQT CONVERSION FUNCTIONS
# ============================================================================

def split_pdbqt_models(pdbqt_file: str) -> list[str]:
    """
    Split a multi-model PDBQT file (from AutoDock Vina) into separate model blocks.
    
    Args:
        pdbqt_file: Path to the PDBQT file containing multiple docked poses
        
    Returns:
        List of strings, each containing one MODEL's content
    """
    with open(pdbqt_file, 'r') as f:
        content = f.read()
    
    models = []
    current_model = []
    in_model = False
    
    for line in content.split('\n'):
        if line.startswith('MODEL'):
            in_model = True
            current_model = [line]
        elif line.startswith('ENDMDL'):
            current_model.append(line)
            models.append('\n'.join(current_model))
            current_model = []
            in_model = False
        elif in_model:
            current_model.append(line)
    
    # If no MODEL/ENDMDL markers, treat entire file as single model
    if not models and content.strip():
        models = [content]
    
    return models

def convert_pdbqt_to_sdf_files(pdbqt_file: str, output_dir: Path) -> list[str]:
    """
    Convert a multi-pose PDBQT file to multiple SDF files.
    
    Uses RDKit's AssignBondOrdersFromTemplate to fix bond orders from the
    original ligand SDF, preventing radical/valence issues that cause
    PoseBusters failures.
    """
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdmolops
    
    pdbqt_path = Path(pdbqt_file)
    base_name = pdbqt_path.stem

    models = split_pdbqt_models(pdbqt_file)
    
    if not models:
        print(f"  Warning: No models found in {pdbqt_file}")
        return []
    
    # ----------------------------------------------------------------
    # Try to find the original ligand SDF to use as bond-order template
    # ----------------------------------------------------------------
    # Parse protein__ligand from filename like "ProteinX__LigandY_vina_out"
    stem_clean = base_name.replace("_vina_out", "")
    parts = stem_clean.split("__")
    template_mol = None
    
    if len(parts) == 2:
        protein_name, ligand_name = parts
        # Search common locations for the original ligand SDF
        ligand_search_dirs = [
            Path(wd) / "ligands",
            Path(wd) / "Ligands",
            Path(wd) / "docking_ready_mgltools",
            Path(wd) / "docking_ready_mgltools" / "ligands",
            Path(wd),
        ]
        
        for search_dir in ligand_search_dirs:
            if not search_dir.exists():
                continue
            # Try exact match and common patterns
            for pattern in [
                f"{ligand_name}.sdf",
                f"{ligand_name}_*.sdf",
                f"*{ligand_name}*.sdf",
            ]:
                matches = list(search_dir.glob(pattern))
                if matches:
                    try:
                        template_mol = Chem.MolFromMolFile(str(matches[0]), removeHs=True, sanitize=True)
                        if template_mol is not None:
                            print(f"    Using bond-order template: {matches[0].name}")
                            break
                    except:
                        pass
            if template_mol is not None:
                break
    
    converted_files = []
    
    for i, model_content in enumerate(models, start=1):
        temp_pdbqt = output_dir / f"{base_name}_model{i}.pdbqt"
        temp_pdb = output_dir / f"{base_name}_model{i}.pdb"
        output_sdf = output_dir / f"{base_name}_model{i}.sdf"
        
        # Skip if already converted
        if output_sdf.exists():
            converted_files.append(str(output_sdf))
            continue
        
        # Write single model PDBQT
        with open(temp_pdbqt, 'w') as f:
            f.write(model_content)
        
        mol_final = None
        
        # ----------------------------------------------------------
        # Strategy 1: obabel PDBQT→PDB, then RDKit with template
        # ----------------------------------------------------------
        if template_mol is not None:
            try:
                # First convert PDBQT to PDB (preserves coordinates, no bond order issues)
                result = subprocess.run(
                    ['obabel', str(temp_pdbqt), '-O', str(temp_pdb)],
                    capture_output=True, text=True
                )
                if result.returncode == 0 and temp_pdb.exists():
                    # Load PDB with RDKit (no sanitization yet)
                    raw_mol = Chem.MolFromPDBFile(str(temp_pdb), removeHs=True, sanitize=False)
                    if raw_mol is not None:
                        try:
                            # Assign correct bond orders from the template
                            mol_final = AllChem.AssignBondOrdersFromTemplate(template_mol, raw_mol)
                            Chem.SanitizeMol(mol_final)
                        except Exception as e:
                            print(f"    Template assignment failed for model {i}: {e}")
                            mol_final = None
                # Clean up temp PDB
                if temp_pdb.exists():
                    temp_pdb.unlink()
            except FileNotFoundError:
                pass  # obabel not installed, fall through
        
        # ----------------------------------------------------------
        # Strategy 2: Direct obabel PDBQT→SDF (original method, fallback)
        # ----------------------------------------------------------
        if mol_final is None:
            try:
                result = subprocess.run(
                    ['obabel', str(temp_pdbqt), '-O', str(output_sdf)],
                    capture_output=True, text=True
                )
                if result.returncode == 0 and output_sdf.exists():
                    # Try to fix bond orders with template even on the SDF
                    if template_mol is not None:
                        try:
                            raw_mol = Chem.MolFromMolFile(str(output_sdf), removeHs=True, sanitize=False)
                            if raw_mol is not None:
                                mol_final = AllChem.AssignBondOrdersFromTemplate(template_mol, raw_mol)
                                Chem.SanitizeMol(mol_final)
                        except:
                            # Keep the obabel output as-is
                            converted_files.append(str(output_sdf))
                            if temp_pdbqt.exists():
                                temp_pdbqt.unlink()
                            continue
                    else:
                        converted_files.append(str(output_sdf))
                        if temp_pdbqt.exists():
                            temp_pdbqt.unlink()
                        continue
            except FileNotFoundError:
                # No obabel available
                try:
                    raw_mol = Chem.MolFromPDBFile(str(temp_pdbqt), removeHs=True, sanitize=False)
                    if raw_mol is not None and template_mol is not None:
                        mol_final = AllChem.AssignBondOrdersFromTemplate(template_mol, raw_mol)
                        Chem.SanitizeMol(mol_final)
                    elif raw_mol is not None:
                        try:
                            Chem.SanitizeMol(raw_mol)
                        except:
                            pass
                        mol_final = raw_mol
                except Exception as e:
                    print(f"    RDKit fallback failed for model {i}: {e}")
        
        # ----------------------------------------------------------
        # Write the final molecule to SDF
        # ----------------------------------------------------------
        if mol_final is not None:
            writer = Chem.SDWriter(str(output_sdf))
            writer.write(mol_final)
            writer.close()
            if output_sdf.exists():
                converted_files.append(str(output_sdf))
        elif not output_sdf.exists():
            print(f"    Warning: Could not convert model {i} from {pdbqt_path.name}")
        
        # Clean up temp PDBQT
        if temp_pdbqt.exists():
            temp_pdbqt.unlink()
    
    return converted_files


# ============================================================================
# HELPER FUNCTIONS FOR FILTERED SUBSET
# ============================================================================

def collect_poses_from_filtered_subset(filtered_df: pd.DataFrame) -> list[dict]:
    """
    Collect all individual pose files from the filtered subset DataFrame.
    
    Args:
        filtered_df: DataFrame with columns: docking_tool, protein, ligand, file_path, pose_count
        
    Returns:
        List of dicts with pose information ready for PoseBusters analysis
    """
    all_poses = []
    
    for _, row in filtered_df.iterrows():
        docking_tool = row["docking_tool"]
        protein = row["protein"]
        ligand = row["ligand"]
        file_path = Path(row["file_path"])
        
        if docking_tool == "AutoDock_Vina":
            if file_path.exists() and file_path.suffix == ".pdbqt":
                print(f"  Converting {file_path.name} to SDF...")
                sdf_files = convert_pdbqt_to_sdf_files(str(file_path), converted_dir)
                
                for sdf_file in sdf_files:
                    sdf_path = Path(sdf_file)
                    model_num = sdf_path.stem.split("_model")[-1] if "_model" in sdf_path.stem else "1"
                    
                    all_poses.append({
                        "method": "autodock",
                        "protein": protein,
                        "ligand": ligand,
                        "pose_file": sdf_file,
                        "pose_name": f"{protein}__{ligand}_pose{model_num}",
                        "original_pdbqt": str(file_path),
                        "file_format": "sdf"
                    })
        
        elif docking_tool == "DiffDock":
            if file_path.is_dir():
                for sdf_file in sorted(file_path.glob("**/*.sdf")):
                    confidence = None
                    if "confidence" in sdf_file.stem.lower():
                        match = re.search(r'confidence[_-]?([\d.]+)', sdf_file.stem, re.IGNORECASE)
                        if match:
                            try:
                                confidence = float(match.group(1))
                            except ValueError:
                                pass
                    
                    try:
                        relative_path = sdf_file.relative_to(file_path)
                        pose_name_suffix = str(relative_path)
                    except ValueError:
                        pose_name_suffix = sdf_file.name
                    
                    pose_info = {
                        "method": "diffdock",
                        "protein": protein,
                        "ligand": ligand,
                        "pose_file": str(sdf_file),
                        "pose_name": f"{ligand}__{protein}/{pose_name_suffix}",
                        "file_format": "sdf"
                    }
                    
                    if confidence is not None:
                        pose_info["confidence"] = confidence
                    
                    all_poses.append(pose_info)
        
        elif docking_tool == "EquiBind":
            if file_path.is_dir():
                # Use recursive glob to find SDF files inside site_XX subdirs
                for sdf_file in sorted(file_path.glob("**/*.sdf")):
                    # Build a descriptive pose name including site info if present
                    try:
                        relative_path = sdf_file.relative_to(file_path)
                        pose_name_suffix = str(relative_path)
                    except ValueError:
                        pose_name_suffix = sdf_file.name
                    
                    all_poses.append({
                        "method": "equibind",
                        "protein": protein,
                        "ligand": ligand,
                        "pose_file": str(sdf_file),
                        "pose_name": f"{ligand}__{protein}/{pose_name_suffix}",
                        "file_format": "sdf"
                    })
    
    return all_poses


AVAILABLE PROTEIN PDB FILES
  Orai1WT-MDSnap-Fr300_cleaned.pdb
  Orai1WT-MDSnap-Fr400_cleaned.pdb
  Orai1WT-MDSnap-Fr499_cleaned.pdb
  Orai1WT-START-Fr0_cleaned.pdb
  Original/Orai1WT-MDSnap-Fr300.pdb
  Original/Orai1WT-MDSnap-Fr400.pdb
  Original/Orai1WT-MDSnap-Fr499.pdb
  Original/Orai1WT-START-Fr0.pdb

Total: 8 PDB files found in /home/manndo/master_dev/Orai

--------------------------------------------------------------------------------
PROTEIN FILE RESOLUTION
--------------------------------------------------------------------------------
  Orai1WT-MDSnap-Fr300: ✓ Orai1WT-MDSnap-Fr300.pdb
  Orai1WT-MDSnap-Fr300_cleaned: ✓ Orai1WT-MDSnap-Fr300_cleaned.pdb
  Orai1WT-MDSnap-Fr400: ✓ Orai1WT-MDSnap-Fr400.pdb
  Orai1WT-MDSnap-Fr400_cleaned: ✓ Orai1WT-MDSnap-Fr400_cleaned.pdb
  Orai1WT-MDSnap-Fr499_cleaned: ✓ Orai1WT-MDSnap-Fr499_cleaned.pdb
  Orai1WT-START-Fr0_cleaned: ✓ Orai1WT-START-Fr0_cleaned.pdb

  Resolved: 6/6 proteins


# PoseBuster Config

In [5]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Output directory for results
output_dir = Path(wd) / "posebusters_results"
output_dir.mkdir(exist_ok=True)

# Temporary directory for converted PDBQT files
converted_dir = output_dir / "converted_pdbqt"
converted_dir.mkdir(exist_ok=True)

# Use "dock" mode (with protein for intermolecular checks) or "mol" mode (ligand only)
CONFIG_MODE = "dock"  # Options: "dock" (with protein), "mol" (ligand only)

# PoseBuster Function

In [6]:
def analyze_poses_with_posebusters(poses_list: list[dict], config: str = "mol") -> pd.DataFrame:
    """
    Run PoseBusters validation on a list of poses.
    
    For "dock" mode, resolves the protein PDB file from the proteins_dir and passes
    it as mol_cond for intermolecular distance checks. Falls back to "mol" mode
    if no protein file is found.
    
    Args:
        poses_list: List of dicts with 'pose_file', 'method', 'protein', 'ligand' keys
        config: PoseBusters config mode ("mol" or "dock")
    
    Returns:
        DataFrame with PoseBusters results merged with pose metadata
    """
    if not poses_list:
        print("No poses to analyze!")
        return pd.DataFrame()
    
    # Initialize PoseBusters instances
    if config == "dock":
        print("Initializing PoseBusters in 'dock' mode (with protein)...")
        buster_dock = PoseBusters(config="dock") 
    else:
        print("Initializing PoseBusters in 'mol' mode (ligand only)...")
        buster_mol = PoseBusters(config="mol")
    
    all_results = []
    total = len(poses_list)
    n_dock = 0
    n_mol_fallback = 0
    n_errors = 0
    
    for idx, pose_info in enumerate(poses_list, 1):
        pose_file = pose_info["pose_file"]
        protein_name = pose_info["protein"]
        
        # Progress indicator every 10 poses
        if idx % 10 == 0 or idx == 1:
            print(f"  Processing pose {idx}/{total}...")
        
        # Skip if file doesn't exist
        if not os.path.exists(pose_file):
            print(f"  Warning: File not found: {pose_file}")
            continue
        
        try:
            protein_file = None
            used_mode = config
            
            if config == "dock":
                # Look up protein file from cache
                protein_file = _protein_file_cache.get(protein_name)
                
                if protein_file is None:
                    # Try finding it fresh (in case cache wasn't built)
                    protein_file = find_protein_file(protein_name)
                
                if protein_file is not None and Path(protein_file).exists():
                    # DOCK MODE: pass mol_pred, mol_true, mol_cond as strings (not lists)
                    df = buster_dock.bust(
                        pose_file,         # mol_pred: predicted ligand pose (str)
                        None,              # mol_true: no crystal reference
                        protein_file,      # mol_cond: protein structure (str)
                        full_report=True
                    )
                    n_dock += 1
                    used_mode = "dock"
                else:
                    # FALLBACK to mol mode: no protein available
                    print(f"  Warning: No protein PDB for '{protein_name}' — using 'mol' mode for {Path(pose_file).name}")
                    df = buster_mol.bust(
                        pose_file,
                        None,
                        None,
                        full_report=True
                    )
                    n_mol_fallback += 1
                    used_mode = "mol (fallback)"
            else:
                # Pure "mol" mode: no protein needed
                df = buster_mol.bust(
                    pose_file,
                    None,
                    None,
                    full_report=True
                )
                used_mode = "mol"
            
            # Add metadata columns
            df["docking_method"] = pose_info["method"]
            df["protein"] = protein_name
            df["ligand"] = pose_info["ligand"]
            df["pose_file"] = pose_info["pose_file"]
            df["pose_name"] = pose_info["pose_name"]
            df["file_format"] = pose_info.get("file_format", "sdf")
            df["protein_file_used"] = protein_file if protein_file else "none"
            df["posebusters_mode"] = used_mode
            
            if "confidence" in pose_info:
                df["diffdock_confidence"] = pose_info["confidence"]
            
            all_results.append(df)
            
        except Exception as e:
            n_errors += 1
            print(f"  Error processing {Path(pose_file).name}: {e}")
    
    # Print processing summary
    print(f"\n  Processing complete:")
    print(f"    Total poses processed: {total}")
    if config == "dock":
        print(f"    Dock mode (with protein): {n_dock}")
        print(f"    Mol mode (fallback, no protein): {n_mol_fallback}")
    print(f"    Errors: {n_errors}")
    
    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        return pd.DataFrame()


In [ ]:
# ============================================================================
# MAIN ANALYSIS - USING FILTERED SUBSET ONLY
# ============================================================================
print("\n" + "=" * 80)
print("PoseBusters Pose Validation - FILTERED SUBSET ONLY")
print("=" * 80)

# Check if filtered_poses_df exists from previous cell
if 'filtered_poses_df' not in dir() or filtered_poses_df.empty:
    print("\nERROR: No filtered poses available!")
    print("Please run the filtering cell first to create filtered_poses_df")
    results_df = pd.DataFrame()
else:
    print(f"\nUsing filtered subset with {len(filtered_poses_df)} protein-ligand combinations")
    print(f"Expected total poses: {filtered_poses_df['pose_count'].sum():,}")
    
    # Show breakdown by docking tool
    print("\n" + "-" * 80)
    print("FILTERED SUBSET BREAKDOWN:")
    print("-" * 80)
    for tool in filtered_poses_df["docking_tool"].unique():
        tool_df = filtered_poses_df[filtered_poses_df["docking_tool"] == tool]
        print(f"  {tool}: {len(tool_df)} combinations, ~{tool_df['pose_count'].sum()} poses")
    
    # Collect all poses from the filtered subset
    print("\n" + "-" * 80)
    print("1. Collecting poses from filtered subset...")
    print("-" * 80)
    
    all_poses = collect_poses_from_filtered_subset(filtered_poses_df)
    print(f"\n   TOTAL: {len(all_poses)} individual poses to analyze")
    
    # Count by method
    method_counts = {}
    for pose in all_poses:
        method = pose["method"]
        method_counts[method] = method_counts.get(method, 0) + 1
    
    print("\n   Poses by method:")
    for method, count in sorted(method_counts.items()):
        print(f"      {method}: {count} poses")
    
    # Run PoseBusters analysis
    print(f"\n" + "-" * 80)
    print(f"2. Running PoseBusters validation (config='{CONFIG_MODE}')...")
    print("-" * 80)
    results_df = analyze_poses_with_posebusters(all_poses, config=CONFIG_MODE)
    
    # Save results
    if not results_df.empty:
        output_file = output_dir / "posebusters_filtered_results.csv"
        results_df.to_csv(output_file, index=False)
        print(f"\n" + "-" * 80)
        print(f"3. Results saved to: {output_file}")
        print("-" * 80)
        print(f"   Total rows: {len(results_df)}")
        print(f"\n   Columns: {list(results_df.columns[:15])}...")
        
        # Quick pass/fail summary
        if "posebusters_mode" in results_df.columns:
            print(f"\n   Mode breakdown:")
            print(results_df["posebusters_mode"].value_counts().to_string(header=False))
    else:
        print("\n   No results to save!")

print("\n" + "=" * 80)
print("Done!")
print("=" * 80)


PoseBusters Pose Validation - FILTERED SUBSET ONLY

Using filtered subset with 48 protein-ligand combinations
Expected total poses: 5,390

--------------------------------------------------------------------------------
FILTERED SUBSET BREAKDOWN:
--------------------------------------------------------------------------------
  AutoDock_Vina: 24 combinations, ~720 poses
  EquiBind: 24 combinations, ~4670 poses

--------------------------------------------------------------------------------
1. Collecting poses from filtered subset...
--------------------------------------------------------------------------------
  Converting Orai1WT-MDSnap-Fr400__Synta-66-OPT-Singlet_vina_out.pdbqt to SDF...
    Using bond-order template: Synta-66-OPT-Singlet.sdf
  Converting Orai1WT-START-Fr0_cleaned__gsk7975a-deprot-OPT_vina_out.pdbqt to SDF...
    Using bond-order template: gsk7975a-deprot-OPT.sdf
    Template assignment failed for model 1: No matching found
    Template assignment failed for mode

[09:14:07] WARNING: More than one matching pattern found - picking one

[09:14:07] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one



    Template assignment failed for model 30: No matching found
  Converting Orai1WT-START-Fr0_cleaned__Synta-66-OPT-Singlet_vina_out.pdbqt to SDF...
    Using bond-order template: Synta-66-OPT-Singlet.sdf


[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picking one

[09:14:08] WARNING: More than one matching pattern found - picki

  Converting Orai1WT-MDSnap-Fr400_cleaned__2abp-nh2-OPT_vina_out.pdbqt to SDF...
    Using bond-order template: 2abp-nh2-OPT.sdf


...

  Converting Orai1WT-MDSnap-Fr300_cleaned__gsk7975a-deprot-OPT_vina_out.pdbqt to SDF...
    Using bond-order template: gsk7975a-deprot-OPT.sdf
    Template assignment failed for model 1: No matching found
    Template assignment failed for model 2: No matching found
    Template assignment failed for model 3: No matching found
    Template assignment failed for model 4: No matching found
    Template assignment failed for model 5: No matching found
    Template assignment failed for model 6: No matching found
    Template assignment failed for model 7: No matching found
    Template assignment failed for model 8: No matching found
    Template assignment failed for model 9: No matching found
    Template assignment failed for model 10: No matching found
    Template assignment failed for model 11: No matching found
    Template assignment failed for model 12: No matching found
    Template assignment failed for model 13: No matching found
    Template assignment failed for model 14: No

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one



    Template assignment failed for model 29: No matching found
    Template assignment failed for model 30: No matching found
  Converting Orai1WT-MDSnap-Fr300_cleaned__2abp-nh2-OPT_vina_out.pdbqt to SDF...
    Using bond-order template: 2abp-nh2-OPT.sdf


[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:28] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picking one

[09:14:29] WARNING: More than one matching pattern found - picki


   TOTAL: 5390 individual poses to analyze

   Poses by method:
      autodock: 720 poses
      equibind: 4670 poses

--------------------------------------------------------------------------------
2. Running PoseBusters validation (config='dock')...
--------------------------------------------------------------------------------
Initializing PoseBusters in 'dock' mode (with protein)...
  Processing pose 1/5390...


In [ ]:
# Quick diagnostic: which methods fail which tests?
df_pb = pd.read_csv("/home/manndo/master_dev/posebusters_results/posebusters_filtered_results.csv")

bottleneck_tests = ['no_radicals', 'non-aromatic_ring_non-flatness', 'internal_steric_clash']
for test in bottleneck_tests:
    if test in df_pb.columns:
        print(f"\n{test} pass rate by method:")
        print(df_pb.groupby('docking_method')[test].mean().round(3) * 100)

In [ ]:
"""
CELL: Copy PoseBusters-Proved Poses to Organized Folders
==========================================================
Copy all poses that passed ALL PoseBusters tests into:
  posebuster_proved/
    ├── autodock_vina/
    ├── diffdock/
    └── equibind/
"""

import pandas as pd
import shutil
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================
wd = Path("/home/manndo/master_dev")
posebusters_results_dir = wd / "posebusters_results"
output_base = wd / "posebuster_proved"

# Subfolders for each method
folders = {
    "AutoDock_Vina": output_base / "autodock_vina",
    "DiffDock":      output_base / "diffdock",
    "EquiBind":      output_base / "equibind",
}

# ============================================================================
# STEP 1: LOAD POSEBUSTERS RESULTS
# ============================================================================

# Try to find the PoseBusters results CSV
# Common filenames from posebusters output
# candidate_files = [
#     posebusters_results_dir / "posebusters_results_all.csv",
#     posebusters_results_dir / "posebusters_results.csv",
#     posebusters_results_dir / "all_results.csv",
#     posebusters_results_dir / "posebusters_combined_results.csv",
#     posebusters_results_dir / "combined_results.csv",
# ]
# for f in candidate_files:
#     if f.exists():
#         results_csv = f
#         break

# Also search for any CSV in the results directory
results_csv = "/home/manndo/master_dev/posebusters_results/posebusters_filtered_results.csv"

if results_csv is None:
    # Search for any CSV that might contain posebusters results
    csvs = list(posebusters_results_dir.glob("*.csv"))
    if csvs:
        print("Available CSVs in posebusters_results/:")
        for c in csvs:
            print(f"  {c.name}")
        # Try to find one with 'result' in the name
        for c in csvs:
            if 'result' in c.name.lower() or 'posebust' in c.name.lower():
                results_csv = c
                break
        if results_csv is None:
            results_csv = csvs[0]  # fallback to first CSV
        print(f"\nUsing: {results_csv.name}")
    else:
        raise FileNotFoundError(
            f"No PoseBusters results CSV found in {posebusters_results_dir}.\n"
            f"Please check the directory or update the path."
        )

print("=" * 100)
print(f"Loading PoseBusters results from: {results_csv}")
print("=" * 100)

df_pb = pd.read_csv(results_csv)
print(f"\nTotal entries: {len(df_pb)}")
print(f"Columns: {list(df_pb.columns)}")

# ============================================================================
# STEP 2: IDENTIFY TEST COLUMNS AND FILTER PASSED POSES
# ============================================================================

# PoseBusters test columns are typically boolean (True/False or 1/0)
# They usually exclude metadata columns like file_path, method, protein, ligand, etc.
metadata_cols = {
    'file_path', 'filepath', 'file', 'path', 'sdf_file', 'sdf_path',
    'method', 'docking_method', 'protein', 'ligand', 'pose_rank', 'rank',
    'molecule', 'mol_name', 'name', 'complex', 'protein_path', 'ligand_path',
    'mol_pred', 'mol_true', 'mol_cond',
}

# Columns to exclude: not real pass/fail tests
# - mol_true_loaded / mol_cond_loaded: always False in "mol" mode (no reference/protein provided)
# - number_* columns: these are integer counts, not boolean tests
# - num_h_added: count of hydrogens added, not a test
exclude_cols = {
    'mol_true_loaded', 'mol_cond_loaded',  # Always False in "mol" mode
    'number_short_outlier_bonds', 'number_long_outlier_bonds',
    'number_outlier_angles', 'number_clashes',
    'number_non-aromatic_rings_pass', 'number_aromatic_rings_pass',
    'number_non-aromatic_rings_checked', 'number_aromatic_rings_checked',
    'number_double_bonds_checked', 'number_double_bonds_pass',
    'number_valid_bonds', 'number_valid_angles', 'number_valid_noncov_pairs',
    'number_noncov_pairs', 'number_bonds', 'number_angles',
    'num_h_added',
}

# Identify test columns (boolean or 0/1 columns that aren't metadata or excluded)
test_cols = []
for col in df_pb.columns:
    col_lower = col.lower().strip()
    if col_lower in metadata_cols:
        continue
    if col in exclude_cols or col_lower in exclude_cols:
        continue
    # Also skip any column starting with "number_" or "num_" — these are counts, not tests
    if col_lower.startswith('number_') or col_lower.startswith('num_'):
        continue
    # Check if column is boolean-like
    unique_vals = df_pb[col].dropna().unique()
    if set(unique_vals).issubset({True, False, 1, 0, 1.0, 0.0, 'True', 'False', 'true', 'false'}):
        test_cols.append(col)

if not test_cols:
    # Fallback: try all columns that are bool dtype
    test_cols = [c for c in df_pb.columns if df_pb[c].dtype == bool]

if not test_cols:
    print("\nWARNING: Could not auto-detect test columns. Showing all columns and dtypes:")
    for col in df_pb.columns:
        print(f"  {col}: {df_pb[col].dtype}, unique values: {df_pb[col].nunique()}")
    raise ValueError("Cannot identify PoseBusters test columns automatically. "
                     "Please update the script with the correct column names.")

print(f"\nIdentified {len(test_cols)} PoseBusters test columns:")
for tc in test_cols:
    n_pass = df_pb[tc].astype(bool).sum() if df_pb[tc].dtype != object else (df_pb[tc].isin([True, 'True', 'true', 1])).sum()
    print(f"  {tc}: {n_pass}/{len(df_pb)} passed ({100*n_pass/len(df_pb):.1f}%)")

# Show which columns were excluded and why
excluded_found = [col for col in df_pb.columns if col in exclude_cols or col.lower() in exclude_cols 
                  or col.lower().startswith('number_') or col.lower().startswith('num_')]
if excluded_found:
    print(f"\nExcluded {len(excluded_found)} non-test columns (counts, always-false indicators):")
    for ec in excluded_found:
        print(f"  ✗ {ec}")

# Convert test columns to boolean
for tc in test_cols:
    if df_pb[tc].dtype == object:
        df_pb[tc] = df_pb[tc].map({'True': True, 'true': True, 'False': False, 'false': False})
    df_pb[tc] = df_pb[tc].astype(bool)

# Filter: keep only rows where ALL tests passed
df_pb["all_passed"] = df_pb[test_cols].all(axis=1)
df_passed = df_pb[df_pb["all_passed"]].copy()

print(f"\n{'=' * 100}")
print(f"POSES THAT PASSED ALL {len(test_cols)} POSEBUSTERS TESTS: "
      f"{len(df_passed)} / {len(df_pb)} ({100*len(df_passed)/len(df_pb):.1f}%)")
print(f"{'=' * 100}")

# Show which tests are the bottlenecks (lowest pass rates)
print(f"\nBottleneck tests (lowest pass rates):")
pass_rates = {tc: df_pb[tc].sum() / len(df_pb) * 100 for tc in test_cols}
for tc, rate in sorted(pass_rates.items(), key=lambda x: x[1]):
    if rate < 99.0:
        print(f"  {tc}: {rate:.1f}%")

# ============================================================================
# STEP 3: IDENTIFY FILE PATH AND METHOD COLUMNS
# ============================================================================

# Find the file path column
file_col = None
for candidate in ['file_path', 'filepath', 'sdf_file', 'sdf_path', 'path', 'file',
                   'mol_pred', 'File', 'FILE_PATH', 'pose_file']:
    if candidate in df_passed.columns:
        file_col = candidate
        break
# Fallback: first column containing 'path' or 'file'
if file_col is None:
    for col in df_passed.columns:
        if 'path' in col.lower() or 'file' in col.lower():
            file_col = col
            break

# Find the method column
method_col = None
for candidate in ['method', 'docking_method', 'Method', 'DOCKING_METHOD']:
    if candidate in df_passed.columns:
        method_col = candidate
        break

if file_col is None:
    print("\nAvailable columns:", list(df_passed.columns))
    raise ValueError("Cannot find file path column. Please update the script.")

print(f"\nFile path column: '{file_col}'")
print(f"Method column:    '{method_col}'" if method_col else "Method column: NOT FOUND (will infer from path)")

# ============================================================================
# STEP 4: DETERMINE METHOD FROM FILE PATH (IF NO METHOD COLUMN)
# ============================================================================

def infer_method(filepath):
    """Infer docking method from file path."""
    fp = str(filepath).lower()
    if any(x in fp for x in ['autodock', 'vina', 'docking_ready', 'pdbqt', '_vina_']):
        return "AutoDock_Vina"
    elif any(x in fp for x in ['diffdock', 'diff_dock']):
        return "DiffDock"
    elif any(x in fp for x in ['equibind', 'equi_bind']):
        return "EquiBind"
    # Check converted_pdbqt directory (AutoDock Vina converted files)
    if 'converted_pdbqt' in fp or 'converted' in fp:
        return "AutoDock_Vina"
    return "Unknown"


if method_col:
    df_passed["_method"] = df_passed[method_col]
else:
    df_passed["_method"] = df_passed[file_col].apply(infer_method)

# Normalize method names
method_normalize = {
    'autodock_vina': 'AutoDock_Vina',
    'autodock vina': 'AutoDock_Vina',
    'autodock': 'AutoDock_Vina',
    'vina': 'AutoDock_Vina',
    'diffdock': 'DiffDock',
    'equibind': 'EquiBind',
}
df_passed["_method"] = df_passed["_method"].apply(
    lambda x: method_normalize.get(str(x).lower().strip(), str(x))
)

print(f"\nPassed poses by method:")
for method, count in df_passed["_method"].value_counts().items():
    print(f"  {method}: {count}")

# ============================================================================
# STEP 5: CREATE FOLDER STRUCTURE AND COPY FILES
# ============================================================================

# Create output directories
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)

copied_count = {"AutoDock_Vina": 0, "DiffDock": 0, "EquiBind": 0, "Unknown": 0}
skipped_count = {"not_found": 0, "unknown_method": 0}

print(f"\n{'=' * 100}")
print("COPYING POSEBUSTERS-PROVED POSES...")
print(f"{'=' * 100}")

for _, row in df_passed.iterrows():
    src_path = Path(str(row[file_col]))
    method = row["_method"]
    
    # Handle relative paths
    if not src_path.is_absolute():
        src_path = wd / src_path
    
    if not src_path.exists():
        # Try a few common path resolutions
        alt_paths = [
            posebusters_results_dir / src_path.name,
            posebusters_results_dir / "converted_pdbqt" / src_path.name,
            wd / "posebusters_results" / "converted_pdbqt" / src_path.name,
        ]
        found = False
        for alt in alt_paths:
            if alt.exists():
                src_path = alt
                found = True
                break
        if not found:
            skipped_count["not_found"] += 1
            continue
    
    # Determine destination folder
    if method in folders:
        dest_dir = folders[method]
    else:
        skipped_count["unknown_method"] += 1
        print(f"  WARNING: Unknown method '{method}' for {src_path.name}")
        continue
    
    # Create a meaningful filename preserving protein/ligand info
    dest_file = dest_dir / src_path.name
    
    # Handle duplicates by adding a suffix
    if dest_file.exists():
        stem = dest_file.stem
        suffix = dest_file.suffix
        counter = 1
        while dest_file.exists():
            dest_file = dest_dir / f"{stem}_dup{counter}{suffix}"
            counter += 1
    
    shutil.copy2(str(src_path), str(dest_file))
    copied_count[method] += 1

# ============================================================================
# STEP 6: REPORT
# ============================================================================

print(f"\n{'=' * 100}")
print("COPY COMPLETE — SUMMARY")
print(f"{'=' * 100}")
print(f"\n  Output directory: {output_base}")
print(f"\n  Files copied per method:")
total_copied = 0
for method, count in copied_count.items():
    if count > 0:
        folder_path = folders.get(method, "N/A")
        print(f"    {method:>20}: {count:>4} files  →  {folder_path}")
        total_copied += count

print(f"\n  Total files copied: {total_copied}")

if skipped_count["not_found"] > 0:
    print(f"\n  WARNING: {skipped_count['not_found']} files skipped (source not found)")
if skipped_count["unknown_method"] > 0:
    print(f"  WARNING: {skipped_count['unknown_method']} files skipped (unknown method)")

# Verify directory contents
print(f"\n  Directory contents:")
for method_name, folder_path in folders.items():
    n_files = len(list(folder_path.glob("*")))
    print(f"    {folder_path.relative_to(wd)}: {n_files} files")

# Save a manifest CSV
manifest = df_passed[[file_col, "_method"] + test_cols].copy()
manifest.rename(columns={"_method": "docking_method"}, inplace=True)
manifest_path = output_base / "posebuster_proved_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"\n  Manifest saved: {manifest_path}")

print(f"\n{'=' * 100}")

In [ ]:
"""
Graphical Representation of PoseBusters Results
=================================================
Comprehensive visualizations of validation test pass rates,
per-method breakdowns, and overall quality profiles.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

# ============================================================================
# LOAD DATA
# ============================================================================
pb_csv = "/home/manndo/master_dev/posebusters_results/posebusters_filtered_results.csv"
output_dir = Path("/home/manndo/master_dev/posebusters_results")
df = pd.read_csv(pb_csv)

# Identify boolean test columns (same logic as manifest cell)
exclude_cols = {
    'mol_true_loaded', 'mol_cond_loaded',
    'number_short_outlier_bonds', 'number_long_outlier_bonds',
    'number_outlier_angles', 'number_clashes',
    'number_non-aromatic_rings_pass', 'number_aromatic_rings_pass',
    'number_non-aromatic_rings_checked', 'number_aromatic_rings_checked',
    'number_double_bonds_checked', 'number_double_bonds_pass',
    'number_valid_bonds', 'number_valid_angles', 'number_valid_noncov_pairs',
    'number_noncov_pairs', 'number_bonds', 'number_angles',
    'num_h_added',
}
metadata_cols = {
    'file_path', 'filepath', 'file', 'path', 'sdf_file', 'sdf_path',
    'method', 'docking_method', 'protein', 'ligand', 'pose_rank', 'rank',
    'molecule', 'mol_name', 'name', 'complex', 'protein_path', 'ligand_path',
    'mol_pred', 'mol_true', 'mol_cond',
}

test_cols = []
for col in df.columns:
    cl = col.lower().strip()
    if cl in metadata_cols or col in exclude_cols:
        continue
    if cl.startswith('number_') or cl.startswith('num_'):
        continue
    uv = set(df[col].dropna().unique())
    if uv.issubset({True, False, 1, 0, 1.0, 0.0, 'True', 'False', 'true', 'false'}):
        test_cols.append(col)

for tc in test_cols:
    if df[tc].dtype == object:
        df[tc] = df[tc].map({'True': True, 'true': True, 'False': False, 'false': False})
    df[tc] = df[tc].astype(bool)

df["all_passed"] = df[test_cols].all(axis=1)

# Friendly display names for tests
TEST_DISPLAY = {
    'mol_pred_loaded': 'Molecule Loaded',
    'sanitization': 'Sanitization',
    'inchi_convertible': 'InChI Convertible',
    'all_atoms_connected': 'All Atoms Connected',
    'no_radicals': 'No Radicals',
    'bond_lengths': 'Bond Lengths',
    'bond_angles': 'Bond Angles',
    'internal_steric_clash': 'No Steric Clash',
    'aromatic_ring_flatness': 'Aromatic Flatness',
    'non-aromatic_ring_non-flatness': 'Non-Arom. Ring Shape',
    'double_bond_flatness': 'Double Bond Flatness',
    'internal_energy': 'Internal Energy',
    'passes_valence_checks': 'Valence Checks',
    'passes_kekulization': 'Kekulization',
    'no_radicals_before_sanitization': 'No Pre-Sanit. Radicals',
}

# Method colors
METHOD_COLORS = {
    'autodock': '#3498db',
    'diffdock': '#e74c3c',
    'equibind': '#2ecc71',
}

methods = sorted(df['docking_method'].unique())
n_total = len(df)
n_passed = df['all_passed'].sum()

print(f"PoseBusters Results: {n_total} total poses, {n_passed} passed all tests ({n_passed/n_total*100:.1f}%)")
for m in methods:
    g = df[df['docking_method'] == m]
    print(f"  {m}: {g['all_passed'].sum()}/{len(g)} passed ({g['all_passed'].mean()*100:.1f}%)")

# ============================================================================
# FIGURE 1: Overall pass rate heatmap (methods × tests)
# ============================================================================
# Only show tests that have at least one failure (skip trivially 100% tests)
variable_tests = [t for t in test_cols if df[t].mean() < 1.0]
trivial_tests = [t for t in test_cols if t not in variable_tests]

pass_rates = df.groupby('docking_method')[test_cols].mean() * 100
pass_rates_var = pass_rates[variable_tests] if variable_tests else pass_rates

fig1, ax1 = plt.subplots(figsize=(max(10, len(test_cols) * 0.8), 4))

# Custom red-yellow-green colormap
cmap = LinearSegmentedColormap.from_list('ryg', ['#e74c3c', '#f39c12', '#27ae60'])

im = ax1.imshow(pass_rates_var.values, cmap=cmap, aspect='auto', vmin=50, vmax=100)

ax1.set_yticks(range(len(pass_rates_var.index)))
ax1.set_yticklabels(pass_rates_var.index, fontsize=11, fontweight='bold')
ax1.set_xticks(range(len(pass_rates_var.columns)))
xlabels = [TEST_DISPLAY.get(c, c.replace('_', ' ').title()) for c in pass_rates_var.columns]
ax1.set_xticklabels(xlabels, rotation=45, ha='right', fontsize=10)

# Annotate cells
for i in range(pass_rates_var.shape[0]):
    for j in range(pass_rates_var.shape[1]):
        val = pass_rates_var.values[i, j]
        color = 'white' if val < 75 else 'black'
        ax1.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=10,
                fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=ax1, shrink=0.8, pad=0.02)
cbar.set_label('Pass Rate (%)', fontsize=11)

if trivial_tests:
    trivial_str = ', '.join(TEST_DISPLAY.get(t, t) for t in trivial_tests)
    ax1.set_xlabel(f'Tests at 100% (not shown): {trivial_str}', fontsize=8, style='italic')

ax1.set_title('PoseBusters Test Pass Rates by Docking Method', fontsize=14, fontweight='bold', pad=12)
fig1.tight_layout()
fig1.savefig(output_dir / "pb_passrate_heatmap.png", dpi=200, bbox_inches='tight')
plt.show()

# ============================================================================
# FIGURE 2: Stacked bar — pass / fail counts per method
# ============================================================================
fig2, ax2 = plt.subplots(figsize=(7, 5))

counts = df.groupby('docking_method')['all_passed'].value_counts().unstack(fill_value=0)
if True not in counts.columns:
    counts[True] = 0
if False not in counts.columns:
    counts[False] = 0
counts = counts[[True, False]].rename(columns={True: 'Passed All', False: 'Failed ≥1'})

bars_pass = ax2.bar(range(len(counts)), counts['Passed All'],
                    color=[METHOD_COLORS.get(m, '#888') for m in counts.index],
                    edgecolor='white', linewidth=1.5, label='Passed All Tests')
bars_fail = ax2.bar(range(len(counts)), counts['Failed ≥1'],
                    bottom=counts['Passed All'],
                    color=[METHOD_COLORS.get(m, '#888') for m in counts.index],
                    alpha=0.3, edgecolor='white', linewidth=1.5, hatch='///',
                    label='Failed ≥1 Test')

# Annotate
for i, (m, row) in enumerate(counts.iterrows()):
    total = row.sum()
    pct = row['Passed All'] / total * 100
    ax2.text(i, total + 5, f'{int(row["Passed All"])}/{int(total)}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_xticks(range(len(counts)))
ax2.set_xticklabels(counts.index, fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Poses', fontsize=12)
ax2.set_title('PoseBusters Validation Summary by Docking Method', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.set_ylim(0, counts.sum(axis=1).max() * 1.25)
ax2.grid(axis='y', alpha=0.3)
fig2.tight_layout()
fig2.savefig(output_dir / "pb_pass_fail_bars.png", dpi=200, bbox_inches='tight')
plt.show()

# ============================================================================
# FIGURE 3: Per-test failure rate comparison (grouped bar chart)
#            Only shows tests with failures
# ============================================================================
if variable_tests:
    fail_rates = (1 - df.groupby('docking_method')[variable_tests].mean()) * 100

    fig3, ax3 = plt.subplots(figsize=(max(10, len(variable_tests) * 1.5), 5))

    x = np.arange(len(variable_tests))
    width = 0.8 / len(methods)

    for i, m in enumerate(methods):
        vals = fail_rates.loc[m].values
        bars = ax3.bar(x + i * width - 0.4 + width / 2, vals, width,
                       label=m, color=METHOD_COLORS.get(m, '#888'), edgecolor='white')
        # Label bars > 2%
        for j, v in enumerate(vals):
            if v > 2:
                ax3.text(x[j] + i * width - 0.4 + width / 2, v + 0.5,
                         f'{v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax3.set_xticks(x)
    xlabels3 = [TEST_DISPLAY.get(c, c.replace('_', ' ').title()) for c in variable_tests]
    ax3.set_xticklabels(xlabels3, rotation=40, ha='right', fontsize=10)
    ax3.set_ylabel('Failure Rate (%)', fontsize=12)
    ax3.set_title('PoseBusters Failure Rates by Test and Method', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(axis='y', alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(output_dir / "pb_failure_rates.png", dpi=200, bbox_inches='tight')
    plt.show()

# ============================================================================
# FIGURE 4: Per protein-ligand pass rate (faceted by method)
# ============================================================================
fig4, axes4 = plt.subplots(1, len(methods), figsize=(6 * len(methods), 5), sharey=True)
if len(methods) == 1:
    axes4 = [axes4]

for ax, m in zip(axes4, methods):
    sub = df[df['docking_method'] == m]
    combo_pass = sub.groupby(['protein', 'ligand'])['all_passed'].mean() * 100
    combo_pass = combo_pass.sort_values(ascending=True)

    labels = [f'{p}\n{l}' for p, l in combo_pass.index]
    colors = [('#27ae60' if v >= 75 else '#f39c12' if v >= 50 else '#e74c3c') for v in combo_pass.values]

    ax.barh(range(len(combo_pass)), combo_pass.values, color=colors, edgecolor='white')
    ax.set_yticks(range(len(combo_pass)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Pass Rate (%)', fontsize=11)
    ax.set_title(m, fontsize=13, fontweight='bold', color=METHOD_COLORS.get(m, 'black'))
    ax.set_xlim(0, 105)
    ax.axvline(75, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='x', alpha=0.3)

    for i, v in enumerate(combo_pass.values):
        ax.text(v + 1, i, f'{v:.0f}%', va='center', fontsize=9)

fig4.suptitle('PoseBusters Pass Rate by Protein–Ligand Combination', fontsize=14, fontweight='bold')
fig4.tight_layout()
fig4.savefig(output_dir / "pb_pass_by_combo.png", dpi=200, bbox_inches='tight')
plt.show()

# ============================================================================
# FIGURE 5: Number of tests failed distribution (violin + strip)
# ============================================================================
df['n_failed'] = df[test_cols].apply(lambda row: (~row).sum(), axis=1)

fig5, ax5 = plt.subplots(figsize=(8, 5))

for i, m in enumerate(methods):
    vals = df[df['docking_method'] == m]['n_failed']
    parts = ax5.violinplot([vals], positions=[i], showmedians=True, widths=0.7)
    for pc in parts['bodies']:
        pc.set_facecolor(METHOD_COLORS.get(m, '#888'))
        pc.set_alpha(0.4)
    for key in ['cbars', 'cmins', 'cmaxes', 'cmedians']:
        parts[key].set_color(METHOD_COLORS.get(m, '#888'))
    # Jitter strip
    jitter = np.random.normal(0, 0.08, len(vals))
    ax5.scatter(np.full(len(vals), i) + jitter, vals, s=10, alpha=0.3,
                color=METHOD_COLORS.get(m, '#888'))

ax5.set_xticks(range(len(methods)))
ax5.set_xticklabels(methods, fontsize=12, fontweight='bold')
ax5.set_ylabel('Number of Tests Failed', fontsize=12)
ax5.set_title('Distribution of Failed Tests per Pose', fontsize=14, fontweight='bold')
ax5.set_ylim(-0.5, df['n_failed'].max() + 1)
ax5.grid(axis='y', alpha=0.3)

# Annotate medians
for i, m in enumerate(methods):
    med = df[df['docking_method'] == m]['n_failed'].median()
    ax5.text(i, med + 0.3, f'median={med:.0f}', ha='center', fontsize=9, fontweight='bold')

fig5.tight_layout()
fig5.savefig(output_dir / "pb_nfailed_violin.png", dpi=200, bbox_inches='tight')
plt.show()

print(f"\nAll figures saved to: {output_dir}")
print("  pb_passrate_heatmap.png  — Test pass rate heatmap (methods × tests)")
print("  pb_pass_fail_bars.png    — Pass/fail stacked bars per method")
print("  pb_failure_rates.png     — Per-test failure rates (grouped bars)")
print("  pb_pass_by_combo.png     — Pass rate by protein–ligand combination")
print("  pb_nfailed_violin.png    — Distribution of #tests failed per pose")

# 3 REMAIN

In [ ]:
"""
Cross-Method Pose Comparison: RMSD Analysis in Angstroms
Compare poses from AutoDock Vina, DiffDock, and EquiBind for the same protein-ligand combinations
"""
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import combinations, product
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdMolAlign
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Warning: RDKit not available. Install with: conda install -c conda-forge rdkit")

# ============================================================================
# CONFIGURATION
# ============================================================================
output_dir = Path( wd + "/posebusters_results")
converted_dir = output_dir / "converted_pdbqt"

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_molecule_from_sdf(sdf_file: str):
    """Load a molecule from an SDF file using RDKit"""
    if not RDKIT_AVAILABLE:
        return None
    try:
        mol = Chem.MolFromMolFile(sdf_file, removeHs=False, sanitize=False)
        if mol is not None:
            try:
                Chem.SanitizeMol(mol)
            except:
                pass  # Keep unsanitized
        return mol
    except Exception as e:
        return None


def calculate_rmsd(mol1, mol2, align=True):
    """
    Calculate RMSD between two molecules in Angstroms.
    
    Args:
        mol1, mol2: RDKit molecule objects
        align: If True, align molecules before calculating RMSD (recommended for pose comparison)
        
    Returns:
        RMSD in Angstroms, or None if calculation fails
    """
    if mol1 is None or mol2 is None:
        return None
    
    try:
        # Get heavy atoms only (exclude hydrogens for more robust comparison)
        mol1_heavy = Chem.RemoveHs(mol1)
        mol2_heavy = Chem.RemoveHs(mol2)
        
        # Check atom counts match
        if mol1_heavy.GetNumAtoms() != mol2_heavy.GetNumAtoms():
            return None
        
        if align:
            # Align mol2 to mol1 and get RMSD
            rmsd = rdMolAlign.AlignMol(mol2_heavy, mol1_heavy)
        else:
            # Calculate RMSD without alignment
            conf1 = mol1_heavy.GetConformer()
            conf2 = mol2_heavy.GetConformer()
            
            coords1 = np.array([conf1.GetAtomPosition(i) for i in range(mol1_heavy.GetNumAtoms())])
            coords2 = np.array([conf2.GetAtomPosition(i) for i in range(mol2_heavy.GetNumAtoms())])
            
            rmsd = np.sqrt(np.mean(np.sum((coords1 - coords2)**2, axis=1)))
        
        return rmsd
    except Exception as e:
        return None


def get_centroid(mol):
    """Get the centroid (center of mass) of a molecule"""
    if mol is None:
        return None
    try:
        mol_heavy = Chem.RemoveHs(mol)
        conf = mol_heavy.GetConformer()
        coords = np.array([conf.GetAtomPosition(i) for i in range(mol_heavy.GetNumAtoms())])
        return coords.mean(axis=0)
    except:
        return None


def calculate_centroid_distance(mol1, mol2):
    """Calculate distance between centroids of two molecules"""
    c1 = get_centroid(mol1)
    c2 = get_centroid(mol2)
    if c1 is None or c2 is None:
        return None
    return np.linalg.norm(c1 - c2)


# ============================================================================
# COLLECT ALL SDF POSES BY PROTEIN-LIGAND COMBINATION
# ============================================================================

def collect_all_sdf_poses():
    """Collect all SDF poses organized by protein-ligand combination and method"""
    poses_by_combination = defaultdict(lambda: defaultdict(list))
    
    # AutoDock Vina (converted SDF files)
    if converted_dir.exists():
        for sdf_file in converted_dir.glob("*.sdf"):
            stem = sdf_file.stem
            # Parse: e.g., "Orai1WT-MDSnap-Fr300__2abp-nh2-OPT_vina_out_model1"
            if "_vina_out_model" in stem:
                base = stem.split("_vina_out_model")[0]
                parts = base.split("__")
                if len(parts) == 2:
                    protein, ligand = parts
                    poses_by_combination[(protein, ligand)]["AutoDock_Vina"].append(str(sdf_file))
    
    # DiffDock (native SDF files)
    diffdock_path = Path(diffdock_poses_dir)
    for subdir in diffdock_path.iterdir():
        if not subdir.is_dir():
            continue
        if "_ligand__" in subdir.name or subdir.name in ["prepared_proteins", "converted_pdbqt"]:
            continue
        
        parts = subdir.name.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            for sdf_file in subdir.glob("**/*.sdf"):
                poses_by_combination[(protein, ligand)]["DiffDock"].append(str(sdf_file))
    
    # EquiBind (native SDF files) - exclude pymol variants
    equibind_path = Path(equibind_poses_dir)
    for subdir in equibind_path.iterdir():
        if not subdir.is_dir() or "_pymol" in subdir.name:
            continue
        
        parts = subdir.name.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            for sdf_file in subdir.glob("*.sdf"):
                poses_by_combination[(protein, ligand)]["EquiBind"].append(str(sdf_file))
    
    return poses_by_combination


# ============================================================================
# MAIN COMPARISON ANALYSIS
# ============================================================================

if not RDKIT_AVAILABLE:
    print("ERROR: RDKit is required for pose comparison. Please install it:")
    print("  conda install -c conda-forge rdkit")
else:
    print("=" * 100)
    print("CROSS-METHOD POSE COMPARISON: RMSD Analysis")
    print("=" * 100)
    
    # Collect all poses
    print("\n1. Collecting SDF poses from all methods...")
    poses_by_combo = collect_all_sdf_poses()
    
    # Filter to combinations with poses from multiple methods
    multi_method_combos = {
        k: v for k, v in poses_by_combo.items() 
        if len(v) > 1  # At least 2 different methods
    }
    
    print(f"\n   Total protein-ligand combinations: {len(poses_by_combo)}")
    print(f"   Combinations with multiple methods: {len(multi_method_combos)}")
    
    # Store comparison results
    comparison_results = []
    method_pair_rmsds = defaultdict(list)
    
    print("\n2. Calculating RMSD between poses from different methods...")
    print("-" * 100)
    
    for (protein, ligand), methods_poses in sorted(multi_method_combos.items()):
        methods = list(methods_poses.keys())
        
        if len(methods) < 2:
            continue
        
        print(f"\n   {protein} + {ligand}:")
        for method in methods:
            print(f"      {method}: {len(methods_poses[method])} poses")
        
        # Compare poses between each pair of methods
        for method1, method2 in combinations(methods, 2):
            poses1 = methods_poses[method1]
            poses2 = methods_poses[method2]
            
            # Load first pose from each method as representative
            mol1 = load_molecule_from_sdf(poses1[0])
            mol2 = load_molecule_from_sdf(poses2[0])
            
            if mol1 is None or mol2 is None:
                print(f"      {method1} vs {method2}: Could not load molecules")
                continue
            
            # Calculate RMSD
            rmsd = calculate_rmsd(mol1, mol2, align=True)
            centroid_dist = calculate_centroid_distance(mol1, mol2)
            
            if rmsd is not None:
                method_pair_rmsds[(method1, method2)].append(rmsd)
                comparison_results.append({
                    "Protein": protein,
                    "Ligand": ligand,
                    "Method_1": method1,
                    "Method_2": method2,
                    "RMSD_Angstrom": round(rmsd, 3),
                    "Centroid_Distance_Angstrom": round(centroid_dist, 3) if centroid_dist else None,
                    "Num_Poses_Method1": len(poses1),
                    "Num_Poses_Method2": len(poses2)
                })
                print(f"      {method1} vs {method2}: RMSD = {rmsd:.3f} Å, Centroid Δ = {centroid_dist:.3f} Å" if centroid_dist else f"      {method1} vs {method2}: RMSD = {rmsd:.3f} Å")
    
    # Create results DataFrame
    if comparison_results:
        df_comparison = pd.DataFrame(comparison_results)
        
        print("\n" + "=" * 100)
        print("DETAILED COMPARISON RESULTS")
        print("=" * 100)
        print(df_comparison.to_string(index=False))
        
        # Summary statistics by method pair
        print("\n" + "=" * 100)
        print("SUMMARY: RMSD STATISTICS BY METHOD PAIR (in Angstroms)")
        print("=" * 100)
        
        summary_data = []
        for (m1, m2), rmsds in method_pair_rmsds.items():
            rmsds = np.array(rmsds)
            summary_data.append({
                "Method_Pair": f"{m1} vs {m2}",
                "N_Comparisons": len(rmsds),
                "Mean_RMSD": round(rmsds.mean(), 3),
                "Std_RMSD": round(rmsds.std(), 3),
                "Min_RMSD": round(rmsds.min(), 3),
                "Max_RMSD": round(rmsds.max(), 3),
                "Median_RMSD": round(np.median(rmsds), 3)
            })
        
        df_summary = pd.DataFrame(summary_data)
        print(df_summary.to_string(index=False))
        
        # Interpretation guide
        print("\n" + "=" * 100)
        print("INTERPRETATION GUIDE")
        print("=" * 100)
        print("""
   RMSD Categories (aligned poses):
   • < 1.0 Å  : Very similar poses (nearly identical binding mode)
   • 1-2 Å    : Similar poses (same general binding site, minor variations)
   • 2-4 Å    : Moderately different poses (same pocket, different orientation)
   • > 4 Å    : Significantly different poses (possibly different binding sites)
   
   Centroid Distance:
   • Measures how far apart the binding locations are
   • Large centroid distance + low RMSD = similar shape, different location
   • Small centroid distance + high RMSD = same location, different conformation
        """)
        
        # Save results
        output_file = output_dir / "cross_method_pose_comparison.csv"
        df_comparison.to_csv(output_file, index=False)
        
        summary_file = output_dir / "cross_method_rmsd_summary.csv"
        df_summary.to_csv(summary_file, index=False)
        
        print(f"\n   Detailed results saved to: {output_file}")
        print(f"   Summary saved to: {summary_file}")
        
    else:
        print("\n   No cross-method comparisons could be made.")
        print("   This may be due to:")
        print("   - Different atom counts between methods")
        print("   - Missing or invalid SDF files")
        print("   - No overlapping protein-ligand combinations")

print("\n" + "=" * 100)
print("Done!")
print("=" * 100)

In [ ]:
"""
Visualization: RMSD Distribution and Heatmap
"""
import matplotlib.pyplot as plt
import numpy as np

if 'df_comparison' in dir() and not df_comparison.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # 1. RMSD Distribution by Method Pair
    ax1 = axes[0]
    method_pairs = df_comparison.groupby(["Method_1", "Method_2"])["RMSD_Angstrom"].apply(list)
    
    data_to_plot = []
    labels = []
    for (m1, m2), rmsds in method_pairs.items():
        data_to_plot.append(rmsds)
        labels.append(f"{m1}\nvs\n{m2}")
    
    bp = ax1.boxplot(data_to_plot, labels=labels, patch_artist=True)
    colors = ['#3498db', '#e74c3c', '#2ecc71'][:len(data_to_plot)]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax1.axhline(y=2.0, color='orange', linestyle='--', alpha=0.5, label='2Å threshold')
    ax1.axhline(y=4.0, color='red', linestyle='--', alpha=0.5, label='4Å threshold')
    ax1.set_ylabel("RMSD (Å)")
    ax1.set_title("RMSD Distribution by Method Pair")
    ax1.legend(loc='upper right')
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. RMSD vs Centroid Distance Scatter
    ax2 = axes[1]
    if "Centroid_Distance_Angstrom" in df_comparison.columns:
        valid_data = df_comparison.dropna(subset=["Centroid_Distance_Angstrom"])
        if not valid_data.empty:
            for (m1, m2), group in valid_data.groupby(["Method_1", "Method_2"]):
                ax2.scatter(group["Centroid_Distance_Angstrom"], group["RMSD_Angstrom"], 
                           label=f"{m1} vs {m2}", alpha=0.7, s=80)
            
            ax2.axhline(y=2.0, color='orange', linestyle='--', alpha=0.5)
            ax2.axvline(x=5.0, color='purple', linestyle='--', alpha=0.5)
            ax2.set_xlabel("Centroid Distance (Å)")
            ax2.set_ylabel("RMSD (Å)")
            ax2.set_title("RMSD vs Centroid Distance")
            ax2.legend()
            ax2.grid(alpha=0.3)
    
    # 3. Heatmap: RMSD by Protein-Ligand Combination
    ax3 = axes[2]
    
    # Pivot table for heatmap
    pivot_data = df_comparison.pivot_table(
        values="RMSD_Angstrom", 
        index="Ligand", 
        columns="Protein",
        aggfunc='mean'
    )
    
    if not pivot_data.empty:
        im = ax3.imshow(pivot_data.values, cmap='RdYlGn_r', aspect='auto')
        
        ax3.set_xticks(range(len(pivot_data.columns)))
        ax3.set_yticks(range(len(pivot_data.index)))
        ax3.set_xticklabels(pivot_data.columns, rotation=45, ha='right', fontsize=8)
        ax3.set_yticklabels(pivot_data.index, fontsize=8)
        
        # Add text annotations
        for i in range(len(pivot_data.index)):
            for j in range(len(pivot_data.columns)):
                val = pivot_data.values[i, j]
                if not np.isnan(val):
                    ax3.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8,
                            color='white' if val > 3 else 'black')
        
        cbar = plt.colorbar(im, ax=ax3, shrink=0.8)
        cbar.set_label("Mean RMSD (Å)")
        ax3.set_title("Mean RMSD by Protein-Ligand")
        ax3.set_xlabel("Protein")
        ax3.set_ylabel("Ligand")
    
    plt.tight_layout()
    plt.savefig(output_dir / "cross_method_rmsd_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to: {output_dir / 'cross_method_rmsd_visualization.png'}")
    
else:
    print("No comparison data available for visualization.")

In [ ]:
"""
Top 3 Poses Comparison Table: RMSD Differences Between Docking Methods
Creates an overview showing how the best poses from each method compare in Angstroms
"""
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import combinations, product
import re

# ============================================================================
# COLLECT TOP 3 POSES FROM EACH METHOD
# ============================================================================

def get_pose_rank(filename):
    """Extract rank number from pose filename for sorting"""
    name = Path(filename).stem.lower()
    
    # DiffDock: rank1.sdf, rank2.sdf, rank1_confidence_X.XX.sdf
    match = re.search(r'rank(\d+)', name)
    if match:
        return int(match.group(1))
    
    # AutoDock: model1, model2, etc. (model1 = best)
    match = re.search(r'model(\d+)', name)
    if match:
        return int(match.group(1))
    
    # EquiBind: pose_01.sdf, pose_02.sdf
    match = re.search(r'pose[_-]?(\d+)', name)
    if match:
        return int(match.group(1))
    
    # Generic numbered files
    match = re.search(r'(\d+)', name)
    if match:
        return int(match.group(1))
    
    return 999  # Default for unknown format


def collect_top3_poses():
    """Collect top 3 poses from each method for each protein-ligand combination"""
    poses_by_combo = defaultdict(lambda: defaultdict(list))
    
    # AutoDock Vina (converted SDF files)
    if converted_dir.exists():
        for sdf_file in converted_dir.glob("*.sdf"):
            stem = sdf_file.stem
            if "_vina_out_model" in stem:
                base = stem.split("_vina_out_model")[0]
                parts = base.split("__")
                if len(parts) == 2:
                    protein, ligand = parts
                    poses_by_combo[(protein, ligand)]["AutoDock_Vina"].append(str(sdf_file))
    
    # DiffDock (native SDF files)
    diffdock_path = Path(diffdock_poses_dir)
    for subdir in diffdock_path.iterdir():
        if not subdir.is_dir():
            continue
        if "_ligand__" in subdir.name or subdir.name in ["prepared_proteins", "converted_pdbqt"]:
            continue
        
        parts = subdir.name.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            for sdf_file in subdir.glob("**/*.sdf"):
                poses_by_combo[(protein, ligand)]["DiffDock"].append(str(sdf_file))
    
    # EquiBind (native SDF files) - exclude pymol variants
    equibind_path = Path(equibind_poses_dir)
    for subdir in equibind_path.iterdir():
        if not subdir.is_dir() or "_pymol" in subdir.name:
            continue
        
        parts = subdir.name.split("__")
        if len(parts) == 2:
            ligand, protein = parts
            for sdf_file in subdir.glob("*.sdf"):
                poses_by_combo[(protein, ligand)]["EquiBind"].append(str(sdf_file))
    
    # Sort by rank and take top 3 from each method
    top3_poses = defaultdict(lambda: defaultdict(list))
    for (protein, ligand), methods in poses_by_combo.items():
        for method, files in methods.items():
            sorted_files = sorted(files, key=get_pose_rank)
            top3_poses[(protein, ligand)][method] = sorted_files[:3]
    
    return top3_poses


# ============================================================================
# CALCULATE PAIRWISE RMSD FOR TOP 3 POSES
# ============================================================================

print("=" * 120)
print("TOP 3 POSES COMPARISON: RMSD Between Docking Methods (in Angstroms)")
print("=" * 120)

# Collect top 3 poses
top3_poses = collect_top3_poses()

# Filter to combinations with multiple methods
multi_method_combos = {
    k: v for k, v in top3_poses.items() 
    if len(v) > 1
}

print(f"\nProtein-ligand combinations with multiple methods: {len(multi_method_combos)}")

# Store all comparison results
all_comparisons = []

for (protein, ligand), methods_poses in sorted(multi_method_combos.items()):
    methods = list(methods_poses.keys())
    
    # Compare each pair of methods
    for method1, method2 in combinations(methods, 2):
        poses1 = methods_poses[method1]
        poses2 = methods_poses[method2]
        
        # Calculate RMSD between top poses from each method
        for i, pose1_file in enumerate(poses1, 1):
            mol1 = load_molecule_from_sdf(pose1_file)
            if mol1 is None:
                continue
            
            for j, pose2_file in enumerate(poses2, 1):
                mol2 = load_molecule_from_sdf(pose2_file)
                if mol2 is None:
                    continue
                
                rmsd = calculate_rmsd(mol1, mol2, align=True)
                centroid_dist = calculate_centroid_distance(mol1, mol2)
                
                if rmsd is not None:
                    all_comparisons.append({
                        "Protein": protein,
                        "Ligand": ligand,
                        "Method_1": method1,
                        "Pose_1_Rank": i,
                        "Method_2": method2,
                        "Pose_2_Rank": j,
                        "RMSD_Å": round(rmsd, 2),
                        "Centroid_Δ_Å": round(centroid_dist, 2) if centroid_dist else None,
                    })

# Create DataFrame
df_top3 = pd.DataFrame(all_comparisons)

if not df_top3.empty:
    # ============================================================================
    # OVERVIEW TABLE 1: Best Pose vs Best Pose (Rank 1 vs Rank 1)
    # ============================================================================
    print("\n" + "=" * 120)
    print("TABLE 1: BEST POSE COMPARISON (Rank 1 vs Rank 1)")
    print("=" * 120)
    
    df_rank1 = df_top3[(df_top3["Pose_1_Rank"] == 1) & (df_top3["Pose_2_Rank"] == 1)]
    
    # Pivot to show all method pairs
    rank1_pivot = df_rank1.pivot_table(
        values="RMSD_Å",
        index=["Protein", "Ligand"],
        columns=["Method_1", "Method_2"],
        aggfunc='first'
    )
    
    # Flatten column names
    rank1_pivot.columns = [f"{m1} vs {m2}" for m1, m2 in rank1_pivot.columns]
    rank1_pivot = rank1_pivot.reset_index()
    
    print(rank1_pivot.to_string(index=False))
    
    # ============================================================================
    # OVERVIEW TABLE 2: All Top 3 Comparisons Summary
    # ============================================================================
    print("\n" + "=" * 120)
    print("TABLE 2: TOP 3 POSES DETAILED COMPARISON")
    print("=" * 120)
    
    # Show comparison label
    df_top3["Comparison"] = df_top3.apply(
        lambda r: f"{r['Method_1']}#{r['Pose_1_Rank']} vs {r['Method_2']}#{r['Pose_2_Rank']}", 
        axis=1
    )
    
    # Group by protein-ligand and show summary
    for (protein, ligand), group in df_top3.groupby(["Protein", "Ligand"]):
        print(f"\n{'─'*80}")
        print(f"  {protein} + {ligand}")
        print(f"{'─'*80}")
        
        # Pivot: rows are Method1 poses, columns are Method2 poses
        for (m1, m2), subgroup in group.groupby(["Method_1", "Method_2"]):
            pivot = subgroup.pivot(
                index="Pose_1_Rank",
                columns="Pose_2_Rank",
                values="RMSD_Å"
            )
            pivot.index = [f"{m1} Rank {i}" for i in pivot.index]
            pivot.columns = [f"{m2} Rank {j}" for j in pivot.columns]
            
            print(f"\n  {m1} vs {m2} (RMSD in Å):")
            print(pivot.to_string())
    
    # ============================================================================
    # OVERVIEW TABLE 3: Summary Statistics by Method Pair
    # ============================================================================
    print("\n" + "=" * 120)
    print("TABLE 3: SUMMARY STATISTICS - ALL TOP 3 POSE COMPARISONS")
    print("=" * 120)
    
    summary_stats = df_top3.groupby(["Method_1", "Method_2"]).agg({
        "RMSD_Å": ["count", "mean", "std", "min", "max", "median"]
    }).round(2)
    summary_stats.columns = ["N_Comparisons", "Mean_RMSD", "Std_RMSD", "Min_RMSD", "Max_RMSD", "Median_RMSD"]
    summary_stats = summary_stats.reset_index()
    summary_stats["Method_Pair"] = summary_stats["Method_1"] + " vs " + summary_stats["Method_2"]
    summary_stats = summary_stats[["Method_Pair", "N_Comparisons", "Mean_RMSD", "Std_RMSD", "Min_RMSD", "Max_RMSD", "Median_RMSD"]]
    
    print(summary_stats.to_string(index=False))
    
    # ============================================================================
    # OVERVIEW TABLE 4: Compact Overview with Min/Mean/Max RMSD
    # ============================================================================
    print("\n" + "=" * 120)
    print("TABLE 4: COMPACT OVERVIEW - RMSD RANGE FOR TOP 3 POSES (Min | Mean | Max)")
    print("=" * 120)
    
    compact_data = []
    for (protein, ligand), group in df_top3.groupby(["Protein", "Ligand"]):
        row = {"Protein": protein, "Ligand": ligand}
        
        for (m1, m2), subgroup in group.groupby(["Method_1", "Method_2"]):
            rmsds = subgroup["RMSD_Å"].values
            pair_name = f"{m1}_vs_{m2}"
            row[f"{pair_name}_Min"] = round(rmsds.min(), 2)
            row[f"{pair_name}_Mean"] = round(rmsds.mean(), 2)
            row[f"{pair_name}_Max"] = round(rmsds.max(), 2)
            row[f"{pair_name}_Range"] = f"{rmsds.min():.1f} | {rmsds.mean():.1f} | {rmsds.max():.1f}"
        
        compact_data.append(row)
    
    df_compact = pd.DataFrame(compact_data)
    
    # Show only Range columns for compact view
    range_cols = ["Protein", "Ligand"] + [c for c in df_compact.columns if "_Range" in c]
    df_compact_display = df_compact[range_cols].copy()
    df_compact_display.columns = [c.replace("_Range", "").replace("_vs_", " vs ") for c in df_compact_display.columns]
    
    print(df_compact_display.to_string(index=False))
    
    # ============================================================================
    # INTERPRETATION
    # ============================================================================
    print("\n" + "=" * 120)
    print("INTERPRETATION")
    print("=" * 120)
    print("""
   Format: Min | Mean | Max RMSD (in Angstroms)
   
   Color Coding Guide:
   • Green  (< 2 Å)  : Methods agree well - similar binding poses
   • Yellow (2-4 Å)  : Moderate agreement - similar pocket, different orientation  
   • Red    (> 4 Å)  : Methods disagree - potentially different binding sites
   
   What this means:
   • Low RMSD across all top 3 poses → High confidence in binding mode
   • High RMSD between methods → Binding site uncertain, need experimental validation
   • Large range (Max - Min) → Poses within method are diverse
    """)
    
    # Save results
    output_file = output_dir / "top3_poses_comparison.csv"
    df_top3.to_csv(output_file, index=False)
    
    compact_file = output_dir / "top3_poses_compact_overview.csv"
    df_compact.to_csv(compact_file, index=False)
    
    print(f"\n   Detailed results saved to: {output_file}")
    print(f"   Compact overview saved to: {compact_file}")

else:
    print("\n   No pose comparisons could be made. Check that:")
    print("   - SDF files exist and are valid")
    print("   - Molecules have matching atom counts")
    print("   - Multiple docking methods were used for the same protein-ligand pairs")

print("\n" + "=" * 120)
print("Done!")
print("=" * 120)

In [ ]:
"""
Summary Statistics and Comparison by Docking Method
"""
# Set pandas display options to show full output
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

if not results_df.empty:
    # Key boolean columns from PoseBusters
    bool_cols = [
        'mol_pred_loaded', 'sanitization', 'all_atoms_connected', 
        'bond_lengths', 'bond_angles', 'internal_steric_clash',
        'aromatic_ring_flatness', 'double_bond_flatness',
        'internal_energy', 'passes_all'  # Note: column names may vary
    ]
    
    # Filter to only columns that exist
    available_bool_cols = [c for c in bool_cols if c in results_df.columns]
    
    print("=" * 60)
    print("SUMMARY BY DOCKING METHOD")
    print("=" * 60)
    
    # Group by method and calculate pass rates
    if available_bool_cols:
        summary = results_df.groupby("docking_method")[available_bool_cols].mean() * 100
        print("\nPass rates (%) for each check:")
        print(summary.round(1).to_string())
    else:
        print("No boolean check columns found in results.")
    
    # Count poses by method
    print("\n" + "-" * 60)
    print("Pose counts by method:")
    print(results_df.groupby("docking_method").size().to_string())
    
    # Count poses by protein-ligand combination
    print("\n" + "-" * 60)
    print("Pose counts by protein-ligand-method:")
    pivot_table = results_df.groupby(["docking_method", "protein", "ligand"]).size().unstack(fill_value=0)
    print(pivot_table.to_string())
    
else:
    print("No results available for summary.")

In [ ]:
"""
CELL: Copy PoseBusters-Proved Poses to Organized Folders
==========================================================
Copy all poses that passed ALL PoseBusters tests into:
  posebuster_proved/
    ├── autodock_vina/
    ├── diffdock/
    └── equibind/
"""

import pandas as pd
import shutil
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================
wd = Path("/home/manndo/master_dev")
posebusters_results_dir = wd / "posebusters_results"
output_base = wd / "posebuster_proved"

# Subfolders for each method
folders = {
    "AutoDock_Vina": output_base / "autodock_vina",
    "DiffDock":      output_base / "diffdock",
    "EquiBind":      output_base / "equibind",
}

# ============================================================================
# STEP 1: LOAD POSEBUSTERS RESULTS
# ============================================================================

# Try to find the PoseBusters results CSV
# Common filenames from posebusters output
candidate_files = [
    posebusters_results_dir / "posebusters_results_all.csv",
    posebusters_results_dir / "posebusters_results.csv",
    posebusters_results_dir / "all_results.csv",
    posebusters_results_dir / "posebusters_combined_results.csv",
    posebusters_results_dir / "combined_results.csv",
]

# Also search for any CSV in the results directory
results_csv = None
for f in candidate_files:
    if f.exists():
        results_csv = f
        break

if results_csv is None:
    # Search for any CSV that might contain posebusters results
    csvs = list(posebusters_results_dir.glob("*.csv"))
    if csvs:
        print("Available CSVs in posebusters_results/:")
        for c in csvs:
            print(f"  {c.name}")
        # Try to find one with 'result' in the name
        for c in csvs:
            if 'result' in c.name.lower() or 'posebust' in c.name.lower():
                results_csv = c
                break
        if results_csv is None:
            results_csv = csvs[0]  # fallback to first CSV
        print(f"\nUsing: {results_csv.name}")
    else:
        raise FileNotFoundError(
            f"No PoseBusters results CSV found in {posebusters_results_dir}.\n"
            f"Please check the directory or update the path."
        )

print("=" * 100)
print(f"Loading PoseBusters results from: {results_csv}")
print("=" * 100)

df_pb = pd.read_csv(results_csv)
print(f"\nTotal entries: {len(df_pb)}")
print(f"Columns: {list(df_pb.columns)}")

# ============================================================================
# STEP 2: IDENTIFY TEST COLUMNS AND FILTER PASSED POSES
# ============================================================================

# PoseBusters test columns are typically boolean (True/False or 1/0)
# They usually exclude metadata columns like file_path, method, protein, ligand, etc.
metadata_cols = {
    'file_path', 'filepath', 'file', 'path', 'sdf_file', 'sdf_path',
    'method', 'docking_method', 'protein', 'ligand', 'pose_rank', 'rank',
    'molecule', 'mol_name', 'name', 'complex', 'protein_path', 'ligand_path',
    'mol_pred', 'mol_true', 'mol_cond',
}

# Identify test columns (boolean or 0/1 columns that aren't metadata)
test_cols = []
for col in df_pb.columns:
    col_lower = col.lower().strip()
    if col_lower in metadata_cols:
        continue
    # Check if column is boolean-like
    unique_vals = df_pb[col].dropna().unique()
    if set(unique_vals).issubset({True, False, 1, 0, 1.0, 0.0, 'True', 'False', 'true', 'false'}):
        test_cols.append(col)

if not test_cols:
    # Fallback: try all columns that are bool dtype
    test_cols = [c for c in df_pb.columns if df_pb[c].dtype == bool]

if not test_cols:
    print("\nWARNING: Could not auto-detect test columns. Showing all columns and dtypes:")
    for col in df_pb.columns:
        print(f"  {col}: {df_pb[col].dtype}, unique values: {df_pb[col].nunique()}")
    raise ValueError("Cannot identify PoseBusters test columns automatically. "
                     "Please update the script with the correct column names.")

print(f"\nIdentified {len(test_cols)} PoseBusters test columns:")
for tc in test_cols:
    n_pass = df_pb[tc].astype(bool).sum() if df_pb[tc].dtype != object else (df_pb[tc].isin([True, 'True', 'true', 1])).sum()
    print(f"  {tc}: {n_pass}/{len(df_pb)} passed ({100*n_pass/len(df_pb):.1f}%)")

# Convert test columns to boolean
for tc in test_cols:
    if df_pb[tc].dtype == object:
        df_pb[tc] = df_pb[tc].map({'True': True, 'true': True, 'False': False, 'false': False})
    df_pb[tc] = df_pb[tc].astype(bool)

# Filter: keep only rows where ALL tests passed
df_pb["all_passed"] = df_pb[test_cols].all(axis=1)
df_passed = df_pb[df_pb["all_passed"]].copy()

print(f"\n{'=' * 100}")
print(f"POSES THAT PASSED ALL {len(test_cols)} POSEBUSTERS TESTS: "
      f"{len(df_passed)} / {len(df_pb)} ({100*len(df_passed)/len(df_pb):.1f}%)")
print(f"{'=' * 100}")

# ============================================================================
# STEP 3: IDENTIFY FILE PATH AND METHOD COLUMNS
# ============================================================================

# Find the file path column
file_col = None
for candidate in ['file_path', 'filepath', 'sdf_file', 'sdf_path', 'path', 'file',
                   'mol_pred', 'File', 'FILE_PATH']:
    if candidate in df_passed.columns:
        file_col = candidate
        break
# Fallback: first column containing 'path' or 'file'
if file_col is None:
    for col in df_passed.columns:
        if 'path' in col.lower() or 'file' in col.lower():
            file_col = col
            break

# Find the method column
method_col = None
for candidate in ['method', 'docking_method', 'Method', 'DOCKING_METHOD']:
    if candidate in df_passed.columns:
        method_col = candidate
        break

if file_col is None:
    print("\nAvailable columns:", list(df_passed.columns))
    raise ValueError("Cannot find file path column. Please update the script.")

print(f"\nFile path column: '{file_col}'")
print(f"Method column:    '{method_col}'" if method_col else "Method column: NOT FOUND (will infer from path)")

# ============================================================================
# STEP 4: DETERMINE METHOD FROM FILE PATH (IF NO METHOD COLUMN)
# ============================================================================

def infer_method(filepath):
    """Infer docking method from file path."""
    fp = str(filepath).lower()
    if any(x in fp for x in ['autodock', 'vina', 'docking_ready', 'pdbqt', '_vina_']):
        return "AutoDock_Vina"
    elif any(x in fp for x in ['diffdock', 'diff_dock']):
        return "DiffDock"
    elif any(x in fp for x in ['equibind', 'equi_bind']):
        return "EquiBind"
    # Check converted_pdbqt directory (AutoDock Vina converted files)
    if 'converted_pdbqt' in fp or 'converted' in fp:
        return "AutoDock_Vina"
    return "Unknown"


if method_col:
    df_passed["_method"] = df_passed[method_col]
else:
    df_passed["_method"] = df_passed[file_col].apply(infer_method)

# Normalize method names
method_normalize = {
    'autodock_vina': 'AutoDock_Vina',
    'autodock vina': 'AutoDock_Vina',
    'autodock': 'AutoDock_Vina',
    'vina': 'AutoDock_Vina',
    'diffdock': 'DiffDock',
    'equibind': 'EquiBind',
}
df_passed["_method"] = df_passed["_method"].apply(
    lambda x: method_normalize.get(str(x).lower().strip(), str(x))
)

print(f"\nPassed poses by method:")
for method, count in df_passed["_method"].value_counts().items():
    print(f"  {method}: {count}")

# ============================================================================
# STEP 5: CREATE FOLDER STRUCTURE AND COPY FILES
# ============================================================================

# Create output directories
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)

copied_count = {"AutoDock_Vina": 0, "DiffDock": 0, "EquiBind": 0, "Unknown": 0}
skipped_count = {"not_found": 0, "unknown_method": 0}

print(f"\n{'=' * 100}")
print("COPYING POSEBUSTERS-PROVED POSES...")
print(f"{'=' * 100}")

for _, row in df_passed.iterrows():
    src_path = Path(str(row[file_col]))
    method = row["_method"]
    
    # Handle relative paths
    if not src_path.is_absolute():
        src_path = wd / src_path
    
    if not src_path.exists():
        # Try a few common path resolutions
        alt_paths = [
            posebusters_results_dir / src_path.name,
            posebusters_results_dir / "converted_pdbqt" / src_path.name,
            wd / "posebusters_results" / "converted_pdbqt" / src_path.name,
        ]
        found = False
        for alt in alt_paths:
            if alt.exists():
                src_path = alt
                found = True
                break
        if not found:
            skipped_count["not_found"] += 1
            continue
    
    # Determine destination folder
    if method in folders:
        dest_dir = folders[method]
    else:
        skipped_count["unknown_method"] += 1
        print(f"  WARNING: Unknown method '{method}' for {src_path.name}")
        continue
    
    # Create a meaningful filename preserving protein/ligand info
    dest_file = dest_dir / src_path.name
    
    # Handle duplicates by adding a suffix
    if dest_file.exists():
        stem = dest_file.stem
        suffix = dest_file.suffix
        counter = 1
        while dest_file.exists():
            dest_file = dest_dir / f"{stem}_dup{counter}{suffix}"
            counter += 1
    
    shutil.copy2(str(src_path), str(dest_file))
    copied_count[method] += 1

# ============================================================================
# STEP 6: REPORT
# ============================================================================

print(f"\n{'=' * 100}")
print("COPY COMPLETE — SUMMARY")
print(f"{'=' * 100}")
print(f"\n  Output directory: {output_base}")
print(f"\n  Files copied per method:")
total_copied = 0
for method, count in copied_count.items():
    if count > 0:
        folder_path = folders.get(method, "N/A")
        print(f"    {method:>20}: {count:>4} files  →  {folder_path}")
        total_copied += count

print(f"\n  Total files copied: {total_copied}")

if skipped_count["not_found"] > 0:
    print(f"\n  WARNING: {skipped_count['not_found']} files skipped (source not found)")
if skipped_count["unknown_method"] > 0:
    print(f"  WARNING: {skipped_count['unknown_method']} files skipped (unknown method)")

# Verify directory contents
print(f"\n  Directory contents:")
for method_name, folder_path in folders.items():
    n_files = len(list(folder_path.glob("*")))
    print(f"    {folder_path.relative_to(wd)}: {n_files} files")

# Save a manifest CSV
manifest = df_passed[[file_col, "_method"] + test_cols].copy()
manifest.rename(columns={"_method": "docking_method"}, inplace=True)
manifest_path = output_base / "posebuster_proved_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"\n  Manifest saved: {manifest_path}")

print(f"\n{'=' * 100}")

In [ ]:
"""
Visualization: Compare pass rates across docking methods
"""
import matplotlib.pyplot as plt

if not results_df.empty:
    # Key boolean columns to visualize
    key_checks = [
        'mol_pred_loaded', 'sanitization', 'all_atoms_connected',
        'bond_lengths', 'bond_angles', 'internal_steric_clash',
        'aromatic_ring_flatness', 'double_bond_flatness'
    ]
    
    # Filter to available columns
    available_checks = [c for c in key_checks if c in results_df.columns]
    
    if available_checks:
        # Calculate pass rates by method
        pass_rates = results_df.groupby("docking_method")[available_checks].mean() * 100
        
        # Create bar chart
        fig, ax = plt.subplots(figsize=(12, 6))
        pass_rates.T.plot(kind='bar', ax=ax, width=0.8)
        
        ax.set_ylabel("Pass Rate (%)")
        ax.set_xlabel("PoseBusters Check")
        ax.set_title("PoseBusters Validation: Pass Rates by Docking Method")
        ax.legend(title="Docking Method", loc='lower right')
        ax.set_ylim(0, 105)
        ax.axhline(y=100, color='green', linestyle='--', alpha=0.3, label='100%')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(output_dir / "posebusters_comparison.png", dpi=150)
        plt.show()
        
        print(f"\nFigure saved to: {output_dir / 'posebusters_comparison.png'}")
    else:
        print("No check columns available for visualization.")
else:
    print("No results available for visualization.")

In [ ]:
# 3D Visualization: RMSD Comparison Across All Docking Method Pairs
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# ============================================================================
# PART 1: 3D HEATMAPS FOR EACH PROTEIN-LIGAND COMBINATION & METHOD PAIR
# ============================================================================

if not df_top3.empty:
    # Get unique protein-ligand combinations and method pairs
    combos = df_top3.groupby(["Protein", "Ligand"]).size().reset_index()[["Protein", "Ligand"]]
    method_pairs = df_top3.groupby(["Method_1", "Method_2"]).size().reset_index()[["Method_1", "Method_2"]]
    
    print(f"Found {len(combos)} protein-ligand combinations")
    print(f"Found {len(method_pairs)} method pairs: {[f'{r.Method_1} vs {r.Method_2}' for _, r in method_pairs.iterrows()]}")
    
    # Calculate number of subplots needed
    n_combos = len(combos)
    n_pairs = len(method_pairs)
    
    # Create figure with subplots for each protein-ligand combination
    fig = plt.figure(figsize=(6 * n_pairs + 2, 5 * n_combos))
    
    subplot_idx = 1
    for combo_idx, (_, combo_row) in enumerate(combos.iterrows()):
        protein = combo_row["Protein"]
        ligand = combo_row["Ligand"]
        
        for pair_idx, (_, pair_row) in enumerate(method_pairs.iterrows()):
            m1, m2 = pair_row["Method_1"], pair_row["Method_2"]
            
            # Filter data for this combination and method pair
            subset = df_top3[
                (df_top3["Protein"] == protein) & 
                (df_top3["Ligand"] == ligand) &
                (df_top3["Method_1"] == m1) & 
                (df_top3["Method_2"] == m2)
            ]
            
            if subset.empty:
                subplot_idx += 1
                continue
            
            # Create RMSD matrix (Method 1 ranks vs Method 2 ranks)
            max_rank_1 = int(subset["Pose_1_Rank"].max())
            max_rank_2 = int(subset["Pose_2_Rank"].max())
            rmsd_matrix = np.full((max_rank_1, max_rank_2), np.nan)
            
            for _, row in subset.iterrows():
                i = int(row["Pose_1_Rank"]) - 1
                j = int(row["Pose_2_Rank"]) - 1
                rmsd_matrix[i, j] = row["RMSD_Å"]
            
            # Create 3D subplot
            ax = fig.add_subplot(n_combos, n_pairs, subplot_idx, projection='3d')
            
            # Create meshgrid
            x_pos = np.arange(max_rank_2)
            y_pos = np.arange(max_rank_1)
            X, Y = np.meshgrid(x_pos, y_pos)
            
            # Flatten for bar3d
            x_flat = X.flatten()
            y_flat = Y.flatten()
            z_flat = np.zeros_like(x_flat, dtype=float)
            dz = rmsd_matrix.flatten()
            
            # Handle NaN values
            valid_mask = ~np.isnan(dz)
            if not valid_mask.any():
                subplot_idx += 1
                continue
            
            dx = dy = 0.6
            
            # Create colors based on RMSD values
            valid_dz = dz[valid_mask]
            norm = plt.Normalize(vmin=np.nanmin(dz), vmax=np.nanmax(dz))
            colors = cm.viridis(norm(dz))
            
            # Plot 3D bars
            ax.bar3d(x_flat[valid_mask], y_flat[valid_mask], z_flat[valid_mask], 
                    dx, dy, dz[valid_mask], color=colors[valid_mask], alpha=0.9, 
                    edgecolor='black', linewidth=0.3)
            
            # Add value labels
            for i in range(max_rank_1):
                for j in range(max_rank_2):
                    if not np.isnan(rmsd_matrix[i, j]):
                        ax.text(j + dx/2, i + dy/2, rmsd_matrix[i, j] + 0.05, 
                                f'{rmsd_matrix[i, j]:.2f}', 
                                ha='center', va='bottom', fontsize=8, fontweight='bold')
            
            # Labels
            ax.set_xlabel(f'{m2} Rank', fontsize=9)
            ax.set_ylabel(f'{m1} Rank', fontsize=9)
            ax.set_zlabel('RMSD (Å)', fontsize=9)
            ax.set_xticks(x_pos + dx/2)
            ax.set_xticklabels([f'R{i+1}' for i in range(max_rank_2)], fontsize=8)
            ax.set_yticks(y_pos + dy/2)
            ax.set_yticklabels([f'R{i+1}' for i in range(max_rank_1)], fontsize=8)
            ax.set_title(f'{protein}\n{ligand}\n{m1} vs {m2}', fontsize=10, fontweight='bold')
            ax.view_init(elev=25, azim=45)
            
            subplot_idx += 1
    
    plt.suptitle('3D RMSD Comparison: All Ranks Across Method Pairs\nper Protein-Ligand Combination', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(output_dir / "3d_rmsd_per_combination.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nFigure saved to: {output_dir / '3d_rmsd_per_combination.png'}")

else:
    print("No comparison data available for visualization.")

In [ ]:
# ============================================================================
# PART 2: OVERALL 3D COMPARISON - Aggregated Across ALL Protein-Ligand Combinations
# ============================================================================

if not df_top3.empty:
    # Get unique method pairs
    method_pairs = df_top3.groupby(["Method_1", "Method_2"]).size().reset_index()[["Method_1", "Method_2"]]
    n_pairs = len(method_pairs)
    
    # Create figure with one 3D plot per method pair
    fig = plt.figure(figsize=(7 * n_pairs, 6))
    
    for pair_idx, (_, pair_row) in enumerate(method_pairs.iterrows()):
        m1, m2 = pair_row["Method_1"], pair_row["Method_2"]
        
        # Filter data for this method pair
        subset = df_top3[(df_top3["Method_1"] == m1) & (df_top3["Method_2"] == m2)]
        
        if subset.empty:
            continue
        
        # Calculate MEAN RMSD matrix across all protein-ligand combinations
        mean_rmsd = subset.groupby(["Pose_1_Rank", "Pose_2_Rank"])["RMSD_Å"].mean().reset_index()
        
        max_rank_1 = int(mean_rmsd["Pose_1_Rank"].max())
        max_rank_2 = int(mean_rmsd["Pose_2_Rank"].max())
        rmsd_matrix = np.full((max_rank_1, max_rank_2), np.nan)
        std_matrix = np.full((max_rank_1, max_rank_2), np.nan)
        
        # Also get standard deviation
        std_rmsd = subset.groupby(["Pose_1_Rank", "Pose_2_Rank"])["RMSD_Å"].std().reset_index()
        
        for _, row in mean_rmsd.iterrows():
            i = int(row["Pose_1_Rank"]) - 1
            j = int(row["Pose_2_Rank"]) - 1
            rmsd_matrix[i, j] = row["RMSD_Å"]
        
        for _, row in std_rmsd.iterrows():
            i = int(row["Pose_1_Rank"]) - 1
            j = int(row["Pose_2_Rank"]) - 1
            std_matrix[i, j] = row["RMSD_Å"]
        
        # Create 3D subplot
        ax = fig.add_subplot(1, n_pairs, pair_idx + 1, projection='3d')
        
        # Create meshgrid
        x_pos = np.arange(max_rank_2)
        y_pos = np.arange(max_rank_1)
        X, Y = np.meshgrid(x_pos, y_pos)
        
        # Flatten for bar3d
        x_flat = X.flatten()
        y_flat = Y.flatten()
        z_flat = np.zeros_like(x_flat, dtype=float)
        dz = rmsd_matrix.flatten()
        
        # Handle NaN values
        valid_mask = ~np.isnan(dz)
        dx = dy = 0.6
        
        # Create colors
        norm = plt.Normalize(vmin=np.nanmin(dz), vmax=np.nanmax(dz))
        colors = cm.plasma(norm(dz))
        
        # Plot 3D bars
        ax.bar3d(x_flat[valid_mask], y_flat[valid_mask], z_flat[valid_mask], 
                dx, dy, dz[valid_mask], color=colors[valid_mask], alpha=0.9, 
                edgecolor='black', linewidth=0.5)
        
        # Add value labels with mean ± std
        for i in range(max_rank_1):
            for j in range(max_rank_2):
                if not np.isnan(rmsd_matrix[i, j]):
                    std_val = std_matrix[i, j] if not np.isnan(std_matrix[i, j]) else 0
                    label = f'{rmsd_matrix[i, j]:.2f}\n±{std_val:.2f}'
                    ax.text(j + dx/2, i + dy/2, rmsd_matrix[i, j] + 0.1, 
                            label, ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        # Labels
        ax.set_xlabel(f'{m2} Rank', fontsize=11, fontweight='bold')
        ax.set_ylabel(f'{m1} Rank', fontsize=11, fontweight='bold')
        ax.set_zlabel('Mean RMSD (Å)', fontsize=11, fontweight='bold')
        ax.set_xticks(x_pos + dx/2)
        ax.set_xticklabels([f'Rank {i+1}' for i in range(max_rank_2)], fontsize=9)
        ax.set_yticks(y_pos + dy/2)
        ax.set_yticklabels([f'Rank {i+1}' for i in range(max_rank_1)], fontsize=9)
        ax.set_title(f'{m1} vs {m2}\n(Averaged across all protein-ligand combinations)', 
                    fontsize=12, fontweight='bold', pad=15)
        ax.view_init(elev=25, azim=45)
        
        # Add colorbar
        sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, shrink=0.6, aspect=15, pad=0.1)
        cbar.set_label('Mean RMSD (Å)', fontsize=10)
    
    plt.suptitle('OVERALL 3D RMSD COMPARISON\nMean RMSD (±Std) Across All Protein-Ligand Combinations', 
                 fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(output_dir / "3d_rmsd_overall_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nFigure saved to: {output_dir / '3d_rmsd_overall_comparison.png'}")
    
    # ============================================================================
    # Print summary table for the overall comparison
    # ============================================================================
    print("\n" + "=" * 100)
    print("OVERALL MEAN RMSD COMPARISON MATRICES (Averaged Across All Combinations)")
    print("=" * 100)
    
    for _, pair_row in method_pairs.iterrows():
        m1, m2 = pair_row["Method_1"], pair_row["Method_2"]
        subset = df_top3[(df_top3["Method_1"] == m1) & (df_top3["Method_2"] == m2)]
        
        if subset.empty:
            continue
        
        mean_pivot = subset.pivot_table(
            values="RMSD_Å", 
            index="Pose_1_Rank", 
            columns="Pose_2_Rank", 
            aggfunc='mean'
        ).round(2)
        
        mean_pivot.index = [f"{m1} Rank {i}" for i in mean_pivot.index]
        mean_pivot.columns = [f"{m2} Rank {j}" for j in mean_pivot.columns]
        
        n_combos = subset.groupby(["Protein", "Ligand"]).ngroups
        
        print(f"\n{m1} vs {m2} (RMSD in Å) - Averaged over {n_combos} protein-ligand combinations:")
        print(mean_pivot.to_string())

else:
    print("No comparison data available for overall visualization.")